In [2]:
from __future__ import annotations

import base64
import json
import random
import re
import time
import wave
from datetime import datetime, timezone
from io import BytesIO
from typing import Any, Literal
from uuid import uuid4

import av
import pandas as pd
from google import genai
from google.api_core.exceptions import (
    Conflict,
    NotFound,
    ResourceExhausted,
    ServiceUnavailable,
)
from google.cloud import bigquery
from google.cloud import storage
from google.cloud import texttospeech
from google.genai import types
from IPython.display import Audio, Image as DisplayImage, Video, display
from PIL import Image as PILImage
from pydantic import BaseModel, ConfigDict, Field, field_validator

In [17]:
PROJECT_ID = "leafy-guide-497515-m4"

GLOBAL_LOCATION = "global"
VIDEO_LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "lost_civilizations_4k_museum_studio"

RUN_TABLE_ID = "museum_runs"
CIVILIZATION_TABLE_ID = "civilizations"
CLAIM_TABLE_ID = "evidence_claims"
MEDIA_TABLE_ID = "museum_media"
REVIEW_TABLE_ID = "media_verification"
CHUNK_TABLE_ID = "museum_knowledge_chunks"
RECOMMENDATION_TABLE_ID = "exhibition_recommendations"

CIVILIZATION_COUNT = 3

REASONING_MODEL_CANDIDATES = [
    "gemini-3.5-flash",
    "gemini-2.5-pro",
    "gemini-2.5-flash",
]

IMAGE_MODEL_CANDIDATES = [
    "gemini-3-pro-image",
    "gemini-3.1-flash-image",
]

VIDEO_MODEL = "veo-3.1-generate-001"

TTS_MODEL_CANDIDATES = [
    "gemini-3.1-flash-tts-preview",
    "gemini-2.5-pro-tts",
    "gemini-2.5-flash-tts",
]

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 1536

IMAGE_RESOLUTION = "4K"
VIDEO_RESOLUTION = "4k"

IMAGE_ASPECT_RATIO = "16:9"
VIDEO_ASPECT_RATIO = "16:9"
VIDEO_DURATION_SECONDS = 8

MIN_4K_WIDTH = 3840
MIN_4K_HEIGHT = 2160

TTS_LANGUAGE_CODE = "en-US"
ARCHAEOLOGIST_VOICE = "Kore"
CURATOR_VOICE = "Charon"

CLAIM_CONFIDENCE_THRESHOLD = 0.55

CHUNK_SIZE_CHARS = 1100
CHUNK_OVERLAP_CHARS = 180
TOP_K_DEFAULT = 10

MAX_RETRIES = 5
BACKOFF_BASE_SECONDS = 8
BACKOFF_MAX_SECONDS = 90

IMAGE_DELAY_SECONDS = 8
AUDIO_DELAY_SECONDS = 5
VIDEO_DELAY_SECONDS = 15
EMBEDDING_DELAY_SECONDS = 1

GCS_IMAGE_PREFIX = "lost-civilizations-4k/images"
GCS_AUDIO_PREFIX = "lost-civilizations-4k/audio"
GCS_VIDEO_PREFIX = "lost-civilizations-4k/videos"
GCS_CLAIM_PREFIX = "lost-civilizations-4k/claim-ledgers"
GCS_MANIFEST_PREFIX = "lost-civilizations-4k/manifests"
GCS_STAGING_PREFIX = "lost-civilizations-4k/staging"
GCS_SUMMARY_PREFIX = "lost-civilizations-4k/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Global location:", GLOBAL_LOCATION)
print("Video location:", VIDEO_LOCATION)
print("Bucket:", BUCKET_NAME)
print("Civilizations:", CIVILIZATION_COUNT)
print("Image resolution:", IMAGE_RESOLUTION)
print("Video resolution:", VIDEO_RESOLUTION)
print("No upscaling or downscaling is allowed.")

Configuration loaded.
Project: leafy-guide-497515-m4
Global location: global
Video location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Civilizations: 3
Image resolution: 4K
Video resolution: 4k
No upscaling or downscaling is allowed.


In [4]:
class MuseumBrief(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        validate_assignment=True,
        str_strip_whitespace=True,
    )

    exhibition_title: str = Field(
        min_length=5,
        max_length=180,
    )

    central_question: str = Field(
        min_length=30,
        max_length=900,
    )

    target_audience: list[str] = Field(
        min_length=2,
        max_length=8,
    )

    required_environments: list[
        Literal[
            "submerged_ocean_city",
            "desert_astronomer_city",
            "geothermal_mountain_city",
        ]
    ]

    evidence_policy: str = Field(
        min_length=50,
        max_length=1200,
    )

    visual_style: str = Field(
        min_length=50,
        max_length=1200,
    )

    historical_constraints: list[str] = Field(
        min_length=5,
        max_length=16,
    )

    media_constraints: list[str] = Field(
        min_length=5,
        max_length=16,
    )

    @field_validator("required_environments")
    @classmethod
    def validate_environments(
        cls,
        value: list[str],
    ) -> list[str]:
        if len(value) != CIVILIZATION_COUNT:
            raise ValueError(
                f"Exactly {CIVILIZATION_COUNT} environments are required."
            )

        if len(value) != len(set(value)):
            raise ValueError(
                "Environment values must be unique."
            )

        return value


class EvidenceClaim(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    claim_id: str = Field(
        min_length=3,
        max_length=80,
    )

    claim_type: Literal[
        "artifact",
        "architecture",
        "infrastructure",
        "environment",
        "social_practice",
        "uncertain_hypothesis",
    ]

    statement: str = Field(
        min_length=20,
        max_length=900,
    )

    evidence_description: str = Field(
        min_length=20,
        max_length=1000,
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
    )

    permitted_visual_elements: list[str] = Field(
        min_length=1,
        max_length=12,
    )

    forbidden_inferences: list[str] = Field(
        min_length=1,
        max_length=12,
    )


class CivilizationDossier(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    civilization_id: str = Field(
        min_length=3,
        max_length=60,
    )

    civilization_name: str = Field(
        min_length=3,
        max_length=140,
    )

    environment_type: Literal[
        "submerged_ocean_city",
        "desert_astronomer_city",
        "geothermal_mountain_city",
    ]

    historical_period: str = Field(
        min_length=10,
        max_length=200,
    )

    geographic_context: str = Field(
        min_length=40,
        max_length=1200,
    )

    discovery_story: str = Field(
        min_length=50,
        max_length=1400,
    )

    architecture_summary: str = Field(
        min_length=60,
        max_length=1400,
    )

    engineering_summary: str = Field(
        min_length=60,
        max_length=1400,
    )

    daily_life_summary: str = Field(
        min_length=60,
        max_length=1400,
    )

    collapse_hypothesis: str = Field(
        min_length=50,
        max_length=1200,
    )

    visual_bible: str = Field(
        min_length=100,
        max_length=2200,
    )

    evidence_claims: list[EvidenceClaim] = Field(
        min_length=6,
        max_length=12,
    )

    image_prompt: str = Field(
        min_length=150,
        max_length=3200,
    )

    video_prompt: str = Field(
        min_length=150,
        max_length=3000,
    )

    audio_style_prompt: str = Field(
        min_length=40,
        max_length=800,
    )

    @field_validator("evidence_claims")
    @classmethod
    def validate_claim_ids(
        cls,
        value: list[EvidenceClaim],
    ) -> list[EvidenceClaim]:
        claim_ids = [
            claim.claim_id
            for claim in value
        ]

        if len(claim_ids) != len(set(claim_ids)):
            raise ValueError(
                "claim_id values must be unique inside a civilization."
            )

        return value


class CivilizationPortfolio(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    exhibition_title: str = Field(
        min_length=5,
        max_length=180,
    )

    curator_statement: str = Field(
        min_length=100,
        max_length=1800,
    )

    civilizations: list[CivilizationDossier]

    @field_validator("civilizations")
    @classmethod
    def validate_civilizations(
        cls,
        value: list[CivilizationDossier],
    ) -> list[CivilizationDossier]:
        if len(value) != CIVILIZATION_COUNT:
            raise ValueError(
                f"Expected exactly {CIVILIZATION_COUNT} civilizations."
            )

        civilization_ids = [
            civilization.civilization_id
            for civilization in value
        ]

        environment_types = [
            civilization.environment_type
            for civilization in value
        ]

        if len(civilization_ids) != len(set(civilization_ids)):
            raise ValueError(
                "civilization_id values must be unique."
            )

        if len(environment_types) != len(set(environment_types)):
            raise ValueError(
                "Every civilization must use a different environment."
            )

        return value


class DialogueTurn(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    speaker: Literal[
        "Archaeologist",
        "Curator",
    ]

    text: str = Field(
        min_length=10,
        max_length=550,
    )


class MuseumDialogue(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    civilization_id: str
    episode_title: str
    style_prompt: str

    turns: list[DialogueTurn] = Field(
        min_length=6,
        max_length=10,
    )

    verified_claim_ids: list[str]
    uncertainty_notice: str

    @field_validator("turns")
    @classmethod
    def validate_dialogue_speakers(
        cls,
        value: list[DialogueTurn],
    ) -> list[DialogueTurn]:
        speakers = {
            turn.speaker
            for turn in value
        }

        if speakers != {
            "Archaeologist",
            "Curator",
        }:
            raise ValueError(
                "Dialogue must contain Archaeologist and Curator."
            )

        return value


class MediaVerification(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    civilization_id: str

    image_summary: str
    video_summary: str
    audio_summary: str

    supported_claim_ids: list[str]
    unsupported_visual_elements: list[str]
    contradictory_elements: list[str]

    image_claim_fidelity_score: int = Field(
        ge=1,
        le=10,
    )

    video_claim_fidelity_score: int = Field(
        ge=1,
        le=10,
    )

    audio_evidence_fidelity_score: int = Field(
        ge=1,
        le=10,
    )

    cross_modal_consistency_score: int = Field(
        ge=1,
        le=10,
    )

    historical_plausibility_score: int = Field(
        ge=1,
        le=10,
    )

    hallucination_risk: Literal[
        "low",
        "medium",
        "high",
    ]

    recommended_corrections: list[str]


class ExhibitionRecommendation(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    recommendation_title: str
    executive_summary: str
    recommended_civilization_order: list[str]
    strongest_civilization_id: str
    strongest_civilization_reason: str
    evidence_quality_observations: list[str]
    media_quality_observations: list[str]
    educational_sequence: list[str]
    visitor_warnings: list[str]
    next_iteration_actions: list[str]
    source_chunk_ids: list[str]
    media_uris_to_use: list[str]


print("Structured Pydantic models created.")

Structured Pydantic models created.


In [5]:
global_genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GLOBAL_LOCATION,
)

video_genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=VIDEO_LOCATION,
)

storage_client = storage.Client(
    project=PROJECT_ID,
)

bucket = storage_client.bucket(
    BUCKET_NAME
)

bucket.reload()

bigquery_client = bigquery.Client(
    project=PROJECT_ID
)

tts_client = texttospeech.TextToSpeechClient()

BUCKET_LOCATION = bucket.location

if BUCKET_LOCATION in {
    "US",
    "EU",
}:
    BIGQUERY_LOCATION = BUCKET_LOCATION
else:
    BIGQUERY_LOCATION = BUCKET_LOCATION.lower()

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("Bucket location:", BUCKET_LOCATION)
print("BigQuery location:", BIGQUERY_LOCATION)

Clients created.
Bucket exists: True
Bucket location: EU
BigQuery location: EU


In [6]:
dataset_ref = bigquery.Dataset(
    f"{PROJECT_ID}.{DATASET_ID}"
)

dataset_ref.location = BIGQUERY_LOCATION

try:
    dataset = bigquery_client.create_dataset(
        dataset_ref
    )

    print(
        "Created dataset:",
        dataset.full_dataset_id,
    )

except Conflict:
    dataset = bigquery_client.get_dataset(
        dataset_ref
    )

    print(
        "Dataset already exists:",
        dataset.full_dataset_id,
    )

run_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{RUN_TABLE_ID}"
)

civilization_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{CIVILIZATION_TABLE_ID}"
)

claim_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{CLAIM_TABLE_ID}"
)

media_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{MEDIA_TABLE_ID}"
)

review_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{REVIEW_TABLE_ID}"
)

chunk_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
)

recommendation_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{RECOMMENDATION_TABLE_ID}"
)

print("Run table:", run_table_ref)
print("Civilization table:", civilization_table_ref)
print("Claim table:", claim_table_ref)
print("Media table:", media_table_ref)
print("Review table:", review_table_ref)
print("Chunk table:", chunk_table_ref)
print("Recommendation table:", recommendation_table_ref)

Created dataset: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio
Run table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.museum_runs
Civilization table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.civilizations
Claim table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.evidence_claims
Media table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.museum_media
Review table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.media_verification
Chunk table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.museum_knowledge_chunks
Recommendation table: leafy-guide-497515-m4.lost_civilizations_4k_museum_studio.exhibition_recommendations


In [7]:
run_schema = [
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "exhibition_title",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "structured_brief_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "structured_portfolio_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "claim_ledger_gcs_uri",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "model_selection_json",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

civilization_schema = [
    bigquery.SchemaField(
        "civilization_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "environment_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "historical_period",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

claim_schema = [
    bigquery.SchemaField(
        "claim_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "claim_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "statement",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "evidence_description",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "confidence",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "permitted_visual_elements",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "forbidden_inferences",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "claim_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

media_schema = [
    bigquery.SchemaField(
        "media_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "media_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "gcs_uri",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "mime_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "generation_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "requested_resolution",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "actual_width",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "actual_height",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "duration_seconds",
        "FLOAT64",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "size_bytes",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "native_4k_verified",
        "BOOLEAN",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "prompt",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

review_schema = [
    bigquery.SchemaField(
        "review_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "review_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "review_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

chunk_schema = [
    bigquery.SchemaField(
        "chunk_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "civilization_id",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "document_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "title",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_number",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "global_chunk_number",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_text",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "embedding_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "embedding_dimension",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "embedding",
        "FLOAT64",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

recommendation_schema = [
    bigquery.SchemaField(
        "recommendation_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "question",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "recommendation_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "used_chunk_ids",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "used_media_uris",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "manifest_gcs_uri",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

print("BigQuery schemas created.")

BigQuery schemas created.


In [8]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(
            table_ref
        )

        print(
            "Table already exists:",
            table.full_table_id,
        )

        return table

    except NotFound:
        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(
            table
        )

        print(
            "Created table:",
            created_table.full_table_id,
        )

        return created_table


run_table = ensure_bigquery_table(
    run_table_ref,
    run_schema,
)

civilization_table = ensure_bigquery_table(
    civilization_table_ref,
    civilization_schema,
)

claim_table = ensure_bigquery_table(
    claim_table_ref,
    claim_schema,
)

media_table = ensure_bigquery_table(
    media_table_ref,
    media_schema,
)

review_table = ensure_bigquery_table(
    review_table_ref,
    review_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

recommendation_table = ensure_bigquery_table(
    recommendation_table_ref,
    recommendation_schema,
)

Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.museum_runs
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.civilizations
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.evidence_claims
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.museum_media
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.media_verification
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.museum_knowledge_chunks
Created table: leafy-guide-497515-m4:lost_civilizations_4k_museum_studio.exhibition_recommendations


In [9]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def parse_gcs_uri(
    gcs_uri: str,
) -> tuple[str, str]:
    if not gcs_uri.startswith("gs://"):
        raise ValueError(
            f"Expected gs:// URI, got: {gcs_uri}"
        )

    without_scheme = gcs_uri.removeprefix(
        "gs://"
    )

    bucket_name, blob_name = without_scheme.split(
        "/",
        1,
    )

    return bucket_name, blob_name


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(
        blob_name
    )

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        BUCKET_NAME,
        blob_name,
    )


def upload_bytes_to_gcs(
    data: bytes,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(
        blob_name
    )

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        BUCKET_NAME,
        blob_name,
    )


def download_gcs_bytes(
    gcs_uri: str,
) -> bytes:
    source_bucket_name, blob_name = parse_gcs_uri(
        gcs_uri
    )

    source_bucket = storage_client.bucket(
        source_bucket_name
    )

    return source_bucket.blob(
        blob_name
    ).download_as_bytes()


def get_gcs_blob_size(
    gcs_uri: str,
) -> int:
    source_bucket_name, blob_name = parse_gcs_uri(
        gcs_uri
    )

    source_bucket = storage_client.bucket(
        source_bucket_name
    )

    blob = source_bucket.blob(
        blob_name
    )

    blob.reload()

    return int(
        blob.size or 0
    )


def safe_slug(
    text: str,
) -> str:
    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        text.lower(),
    )

    return slug.strip("-")[:80] or "asset"


def rows_to_ndjson(
    rows: list[dict[str, Any]],
) -> str:
    return "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        )
        for row in rows
    )

In [10]:
def display_gcs_image(
    gcs_uri: str,
    *,
    width: int = 760,
) -> None:
    display(
        DisplayImage(
            data=download_gcs_bytes(gcs_uri),
            width=width,
        )
    )


def display_gcs_audio(
    gcs_uri: str,
) -> None:
    display(
        Audio(
            data=download_gcs_bytes(gcs_uri),
            autoplay=False,
        )
    )


def display_gcs_video(
    gcs_uri: str,
    *,
    width: int = 760,
) -> None:
    display(
        Video(
            data=download_gcs_bytes(gcs_uri),
            embed=True,
            mimetype="video/mp4",
            width=width,
        )
    )


def inspect_image_bytes(
    image_bytes: bytes,
) -> tuple[int, int]:
    with PILImage.open(
        BytesIO(image_bytes)
    ) as image:
        return image.size


def inspect_video_bytes(
    video_bytes: bytes,
) -> dict[str, Any]:
    with av.open(
        BytesIO(video_bytes)
    ) as container:
        video_stream = next(
            stream
            for stream in container.streams
            if stream.type == "video"
        )

        width = int(
            video_stream.codec_context.width
        )

        height = int(
            video_stream.codec_context.height
        )

        frame_rate = (
            float(video_stream.average_rate)
            if video_stream.average_rate
            else None
        )

        if container.duration is not None:
            duration_seconds = (
                float(container.duration)
                / float(av.time_base)
            )
        elif (
            video_stream.duration is not None
            and video_stream.time_base is not None
        ):
            duration_seconds = float(
                video_stream.duration
                * video_stream.time_base
            )
        else:
            duration_seconds = None

    return {
        "width": width,
        "height": height,
        "frame_rate": frame_rate,
        "duration_seconds": duration_seconds,
    }


def inspect_wav_bytes(
    audio_bytes: bytes,
) -> dict[str, Any]:
    with wave.open(
        BytesIO(audio_bytes),
        "rb",
    ) as wav_file:
        frame_count = wav_file.getnframes()
        frame_rate = wav_file.getframerate()
        channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()

    duration_seconds = (
        frame_count / frame_rate
        if frame_rate
        else None
    )

    return {
        "frame_count": frame_count,
        "frame_rate": frame_rate,
        "channels": channels,
        "sample_width": sample_width,
        "duration_seconds": duration_seconds,
    }


def assert_native_4k(
    *,
    width: int,
    height: int,
    media_label: str,
) -> None:
    if (
        width < MIN_4K_WIDTH
        or height < MIN_4K_HEIGHT
    ):
        raise RuntimeError(
            f"{media_label} is not native 4K. "
            f"Received {width}x{height}; "
            f"required at least "
            f"{MIN_4K_WIDTH}x{MIN_4K_HEIGHT}. "
            "The notebook will not upscale this asset."
        )

In [11]:
RETRYABLE_ERRORS = (
    ResourceExhausted,
    ServiceUnavailable,
)


def is_retryable_error(
    exc: Exception,
) -> bool:
    message = str(exc).lower()

    return (
        isinstance(exc, RETRYABLE_ERRORS)
        or "resource exhausted" in message
        or "429" in message
        or "503" in message
        or "temporarily unavailable" in message
        or "rate limit" in message
        or "quota" in message
    )


def sleep_with_backoff(
    attempt: int,
) -> None:
    delay = min(
        BACKOFF_MAX_SECONDS,
        BACKOFF_BASE_SECONDS * (2 ** (attempt - 1)),
    )

    jitter = random.uniform(
        0.0,
        min(
            3.0,
            delay * 0.1,
        ),
    )

    total_delay = delay + jitter

    print(
        f"Waiting {total_delay:.1f} seconds before retry..."
    )

    time.sleep(
        total_delay
    )


def call_with_backoff(
    function,
    *,
    operation_name: str,
    max_retries: int = MAX_RETRIES,
):
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            return function()

        except Exception as exc:
            if (
                is_retryable_error(exc)
                and attempt < max_retries
            ):
                print(
                    operation_name,
                    "returned a temporary error.",
                )

                print(
                    "Attempt:",
                    attempt,
                    "/",
                    max_retries,
                )

                print(
                    "Error:",
                    str(exc)[:500],
                )

                sleep_with_backoff(
                    attempt
                )

                continue

            raise

    raise RuntimeError(
        f"{operation_name} failed after all retries."
    )

In [12]:
def batch_load_rows_to_bigquery(
    *,
    rows: list[dict[str, Any]],
    table_ref: str,
    schema: list[bigquery.SchemaField],
    table_name: str,
    run_id: str,
) -> dict[str, Any]:
    if not rows:
        raise ValueError(
            f"No rows provided for {table_name}."
        )

    ndjson_text = rows_to_ndjson(
        rows
    )

    staging_blob_name = (
        f"{GCS_STAGING_PREFIX}/"
        f"{run_id}/"
        f"{table_name}.ndjson"
    )

    staging_uri = upload_text_to_gcs(
        ndjson_text,
        blob_name=staging_blob_name,
        content_type="application/x-ndjson",
    )

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=(
            bigquery.SourceFormat.NEWLINE_DELIMITED_JSON
        ),
        write_disposition=(
            bigquery.WriteDisposition.WRITE_APPEND
        ),
    )

    started_at = time.perf_counter()

    load_job = bigquery_client.load_table_from_uri(
        staging_uri,
        table_ref,
        location=dataset.location,
        job_config=job_config,
    )

    load_job.result()

    elapsed_seconds = (
        time.perf_counter()
        - started_at
    )

    result = {
        "table_name": table_name,
        "rows_loaded": len(rows),
        "staging_uri": staging_uri,
        "job_id": load_job.job_id,
        "elapsed_seconds": round(
            elapsed_seconds,
            3,
        ),
    }

    print("=" * 100)
    print(result)

    return result

In [13]:
museum_brief = MuseumBrief(
    exhibition_title=(
        "Echoes Beneath Stone: Three Civilizations That History Forgot"
    ),
    central_question=(
        "How can a museum create compelling visual reconstructions "
        "of lost civilizations while clearly separating archaeological "
        "evidence from interpretation and speculation?"
    ),
    target_audience=[
        "museum visitors",
        "archaeology students",
        "historical designers",
        "technology enthusiasts",
    ],
    required_environments=[
        "submerged_ocean_city",
        "desert_astronomer_city",
        "geothermal_mountain_city",
    ],
    evidence_policy=(
        "Every reconstructed object, structure, social practice or technology "
        "must be connected to an explicit evidence claim. Low-confidence "
        "hypotheses must not be presented as established historical facts."
    ),
    visual_style=(
        "Photorealistic archaeological reconstruction, premium museum "
        "documentary cinematography, historically plausible materials, "
        "realistic weathering, physically believable engineering, atmospheric "
        "natural light and high-detail environmental composition."
    ),
    historical_constraints=[
        "No modern materials unless supported by a claim",
        "No electrical devices unless explicitly supported",
        "No fantasy magic",
        "No unexplained levitation",
        "No impossible architecture",
        "No copied historical monuments",
        "No readable modern text",
        "No real logos",
    ],
    media_constraints=[
        "Exactly three images",
        "Exactly three videos",
        "Exactly three audio files",
        "Images must be native 4K",
        "Videos must be native 4K",
        "No upscaling",
        "No downscaling of generated media",
        "No graphic human remains",
    ],
)

structured_brief_json = (
    museum_brief.model_dump_json(
        indent=2
    )
)

print(structured_brief_json)

{
  "exhibition_title": "Echoes Beneath Stone: Three Civilizations That History Forgot",
  "central_question": "How can a museum create compelling visual reconstructions of lost civilizations while clearly separating archaeological evidence from interpretation and speculation?",
  "target_audience": [
    "museum visitors",
    "archaeology students",
    "historical designers",
    "technology enthusiasts"
  ],
  "required_environments": [
    "submerged_ocean_city",
    "desert_astronomer_city",
    "geothermal_mountain_city"
  ],
  "evidence_policy": "Every reconstructed object, structure, social practice or technology must be connected to an explicit evidence claim. Low-confidence hypotheses must not be presented as established historical facts.",
  "visual_style": "Photorealistic archaeological reconstruction, premium museum documentary cinematography, historically plausible materials, realistic weathering, physically believable engineering, atmospheric natural light and high-deta

In [18]:
CIVILIZATION_OUTLINE_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "exhibition_title",
        "curator_statement",
        "civilizations",
    ],
    "properties": {
        "exhibition_title": {
            "type": "STRING",
            "description": (
                "Title of the museum exhibition."
            ),
        },
        "curator_statement": {
            "type": "STRING",
            "description": (
                "Curatorial explanation of the exhibition."
            ),
        },
        "civilizations": {
            "type": "ARRAY",
            "description": (
                "Exactly three basic civilization concepts."
            ),
            "items": {
                "type": "OBJECT",
                "required": [
                    "civilization_id",
                    "civilization_name",
                    "environment_type",
                    "historical_period",
                    "geographic_context",
                ],
                "properties": {
                    "civilization_id": {
                        "type": "STRING",
                        "description": (
                            "Short unique identifier."
                        ),
                    },
                    "civilization_name": {
                        "type": "STRING",
                        "description": (
                            "Original fictional civilization name."
                        ),
                    },
                    "environment_type": {
                        "type": "STRING",
                        "enum": [
                            "submerged_ocean_city",
                            "desert_astronomer_city",
                            "geothermal_mountain_city",
                        ],
                    },
                    "historical_period": {
                        "type": "STRING",
                        "description": (
                            "Approximate fictional historical period."
                        ),
                    },
                    "geographic_context": {
                        "type": "STRING",
                        "description": (
                            "Geography, climate and physical location."
                        ),
                    },
                },
            },
        },
    },
}


EVIDENCE_CLAIMS_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "civilization_id",
        "claims",
    ],
    "properties": {
        "civilization_id": {
            "type": "STRING",
        },
        "claims": {
            "type": "ARRAY",
            "description": (
                "Six to eight archaeological evidence claims."
            ),
            "items": {
                "type": "OBJECT",
                "required": [
                    "claim_type",
                    "statement",
                    "evidence_description",
                    "confidence",
                    "permitted_visual_elements",
                    "forbidden_inferences",
                ],
                "properties": {
                    "claim_type": {
                        "type": "STRING",
                        "enum": [
                            "artifact",
                            "architecture",
                            "infrastructure",
                            "environment",
                            "social_practice",
                            "uncertain_hypothesis",
                        ],
                    },
                    "statement": {
                        "type": "STRING",
                    },
                    "evidence_description": {
                        "type": "STRING",
                    },
                    "confidence": {
                        "type": "NUMBER",
                        "description": (
                            "Confidence between zero and one."
                        ),
                    },
                    "permitted_visual_elements": {
                        "type": "ARRAY",
                        "items": {
                            "type": "STRING",
                        },
                    },
                    "forbidden_inferences": {
                        "type": "ARRAY",
                        "items": {
                            "type": "STRING",
                        },
                    },
                },
            },
        },
    },
}


CIVILIZATION_DETAILS_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "civilization_id",
        "discovery_story",
        "architecture_summary",
        "engineering_summary",
        "daily_life_summary",
        "collapse_hypothesis",
        "visual_bible",
        "image_prompt",
        "video_prompt",
        "audio_style_prompt",
    ],
    "properties": {
        "civilization_id": {
            "type": "STRING",
        },
        "discovery_story": {
            "type": "STRING",
        },
        "architecture_summary": {
            "type": "STRING",
        },
        "engineering_summary": {
            "type": "STRING",
        },
        "daily_life_summary": {
            "type": "STRING",
        },
        "collapse_hypothesis": {
            "type": "STRING",
        },
        "visual_bible": {
            "type": "STRING",
        },
        "image_prompt": {
            "type": "STRING",
        },
        "video_prompt": {
            "type": "STRING",
        },
        "audio_style_prompt": {
            "type": "STRING",
        },
    },
}

print("Simplified structured-output schemas created.")
print("1. Civilization outline schema")
print("2. Evidence claims schema")
print("3. Civilization details schema")

Simplified structured-output schemas created.
1. Civilization outline schema
2. Evidence claims schema
3. Civilization details schema


In [19]:
def generate_structured_with_fallback(
    *,
    contents,
    response_schema: dict[str, Any],
    operation_name: str,
    temperature: float = 0.25,
    max_output_tokens: int = 8192,
) -> tuple[dict[str, Any], str]:
    errors = []

    for model_id in REASONING_MODEL_CANDIDATES:
        try:
            print("=" * 100)
            print("Operation:", operation_name)
            print("Trying reasoning model:", model_id)

            response = call_with_backoff(
                lambda model_id=model_id: (
                    global_genai_client.models.generate_content(
                        model=model_id,
                        contents=contents,
                        config=types.GenerateContentConfig(
                            response_mime_type="application/json",
                            response_schema=response_schema,
                            temperature=temperature,
                            max_output_tokens=max_output_tokens,
                        ),
                    )
                ),
                operation_name=(
                    f"{operation_name} using {model_id}"
                ),
            )

            if not response.text:
                raise RuntimeError(
                    "Model returned an empty response."
                )

            parsed_response = json.loads(
                response.text
            )

            print("Structured response generated successfully.")
            print("Model:", model_id)

            return (
                parsed_response,
                model_id,
            )

        except Exception as exc:
            error_message = str(exc)

            errors.append(
                {
                    "model": model_id,
                    "error": error_message[:1000],
                }
            )

            print("Model failed:", model_id)
            print(error_message[:1000])

            if "too many states" in error_message.lower():
                print(
                    "The response schema is still too complex "
                    "for this model."
                )

    raise RuntimeError(
        f"{operation_name} failed for all models:\n"
        f"{json.dumps(errors, indent=2, ensure_ascii=False)}"
    )

In [20]:
def generate_civilization_outline(
    brief: MuseumBrief,
) -> tuple[dict[str, Any], str]:
    prompt = f"""
You are an archaeologist, historian and museum curator.

Create the basic outline for exactly
{CIVILIZATION_COUNT} fictional lost civilizations.

Structured museum brief:
{brief.model_dump_json(indent=2)}

Create exactly one civilization for every environment:

1. submerged_ocean_city
2. desert_astronomer_city
3. geothermal_mountain_city

Requirements:
- use a short unique civilization_id
- create an original civilization name
- do not copy a real historical civilization
- use a different environment for every civilization
- provide a plausible fictional historical period
- explain the physical geography and climate
- do not create evidence claims yet
- do not create image, video or audio prompts yet
- follow the provided response schema
"""

    raw_outline, model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=(
                CIVILIZATION_OUTLINE_SCHEMA
            ),
            operation_name=(
                "Generate civilization portfolio outline"
            ),
            temperature=0.4,
            max_output_tokens=4096,
        )
    )

    civilizations = raw_outline.get(
        "civilizations",
        [],
    )

    if len(civilizations) != CIVILIZATION_COUNT:
        raise ValueError(
            f"Expected exactly {CIVILIZATION_COUNT} "
            f"civilizations, received {len(civilizations)}."
        )

    civilization_ids = [
        item["civilization_id"]
        for item in civilizations
    ]

    if len(civilization_ids) != len(
        set(civilization_ids)
    ):
        raise ValueError(
            "civilization_id values must be unique."
        )

    expected_environments = set(
        brief.required_environments
    )

    generated_environments = {
        item["environment_type"]
        for item in civilizations
    }

    if generated_environments != expected_environments:
        raise ValueError(
            f"Expected environments "
            f"{expected_environments}, "
            f"received {generated_environments}."
        )

    return (
        raw_outline,
        model_id,
    )


def generate_evidence_claims(
    civilization_outline: dict[str, Any],
    brief: MuseumBrief,
) -> tuple[list[EvidenceClaim], str]:
    civilization_id = civilization_outline[
        "civilization_id"
    ]

    prompt = f"""
You are an archaeological evidence specialist.

Create between 6 and 8 evidence claims for
the following fictional civilization.

Civilization outline:
{json.dumps(
    civilization_outline,
    indent=2,
    ensure_ascii=False,
)}

Museum evidence policy:
{brief.evidence_policy}

Required claim composition:
- at least one artifact claim
- at least one architecture claim
- at least one infrastructure claim
- at least one environment claim
- at least one social practice claim
- at least one uncertain hypothesis

For every claim:
- explain the physical or contextual evidence
- assign confidence between 0 and 1
- list visual elements that the claim permits
- list conclusions that must not be inferred
- do not create claim_id; identifiers are assigned by Python
- return civilization_id exactly as:
  {civilization_id}
- follow the provided response schema
"""

    raw_claims, model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=(
                EVIDENCE_CLAIMS_SCHEMA
            ),
            operation_name=(
                f"Generate evidence claims for "
                f"{civilization_outline['civilization_name']}"
            ),
            temperature=0.3,
            max_output_tokens=6144,
        )
    )

    if (
        raw_claims.get("civilization_id")
        != civilization_id
    ):
        raise ValueError(
            "Evidence response returned "
            "an incorrect civilization_id."
        )

    raw_claim_items = raw_claims.get(
        "claims",
        [],
    )

    if not 6 <= len(raw_claim_items) <= 8:
        raise ValueError(
            "Expected between 6 and 8 evidence claims, "
            f"received {len(raw_claim_items)}."
        )

    validated_claims = []

    for claim_number, claim_data in enumerate(
        raw_claim_items,
        start=1,
    ):
        normalized_claim_data = {
            **claim_data,
            "claim_id": (
                f"{civilization_id}-claim-"
                f"{claim_number:02d}"
            ),
        }

        validated_claim = EvidenceClaim.model_validate(
            normalized_claim_data
        )

        validated_claims.append(
            validated_claim
        )

    claim_types = {
        claim.claim_type
        for claim in validated_claims
    }

    required_claim_types = {
        "artifact",
        "architecture",
        "infrastructure",
        "environment",
        "social_practice",
        "uncertain_hypothesis",
    }

    missing_claim_types = (
        required_claim_types
        - claim_types
    )

    if missing_claim_types:
        raise ValueError(
            "Missing required claim types: "
            f"{sorted(missing_claim_types)}"
        )

    return (
        validated_claims,
        model_id,
    )


def generate_civilization_details(
    civilization_outline: dict[str, Any],
    evidence_claims: list[EvidenceClaim],
    brief: MuseumBrief,
) -> tuple[dict[str, Any], str]:
    civilization_id = civilization_outline[
        "civilization_id"
    ]

    accepted_claims = [
        claim.model_dump(
            mode="json"
        )
        for claim in evidence_claims
        if (
            claim.confidence
            >= CLAIM_CONFIDENCE_THRESHOLD
        )
    ]

    low_confidence_claims = [
        claim.model_dump(
            mode="json"
        )
        for claim in evidence_claims
        if (
            claim.confidence
            < CLAIM_CONFIDENCE_THRESHOLD
        )
    ]

    prompt = f"""
You are an archaeologist, architectural historian,
museum reconstruction designer and documentary director.

Complete the detailed dossier for this fictional civilization.

Civilization outline:
{json.dumps(
    civilization_outline,
    indent=2,
    ensure_ascii=False,
)}

Accepted evidence claims:
{json.dumps(
    accepted_claims,
    indent=2,
    ensure_ascii=False,
)}

Low-confidence claims:
{json.dumps(
    low_confidence_claims,
    indent=2,
    ensure_ascii=False,
)}

Museum visual style:
{brief.visual_style}

Historical constraints:
{json.dumps(
    brief.historical_constraints,
    indent=2,
    ensure_ascii=False,
)}

Rules:
- ground the reconstruction in accepted evidence claims
- low-confidence claims may only be described as hypotheses
- create a plausible discovery story
- explain architecture and engineering separately
- explain daily life without presenting speculation as fact
- create a cautious collapse hypothesis
- create one consistent visual bible
- the image prompt must describe a native 4K,
  16:9 archaeological reconstruction
- the video prompt must describe one continuous,
  eight-second archaeological documentary shot
- the image and video prompts must use the same
  materials, architecture and environmental conditions
- create a museum dialogue style prompt
- do not include unsupported technology
- do not include fantasy magic
- return civilization_id exactly as:
  {civilization_id}
- follow the provided response schema
"""

    raw_details, model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=(
                CIVILIZATION_DETAILS_SCHEMA
            ),
            operation_name=(
                f"Generate details for "
                f"{civilization_outline['civilization_name']}"
            ),
            temperature=0.35,
            max_output_tokens=8192,
        )
    )

    if (
        raw_details.get("civilization_id")
        != civilization_id
    ):
        raise ValueError(
            "Details response returned "
            "an incorrect civilization_id."
        )

    return (
        raw_details,
        model_id,
    )


def generate_civilization_portfolio(
    brief: MuseumBrief,
) -> tuple[
    CivilizationPortfolio,
    dict[str, Any],
]:
    portfolio_outline, outline_model = (
        generate_civilization_outline(
            brief
        )
    )

    generated_civilizations = []
    claim_models = {}
    detail_models = {}

    for civilization_number, civilization_outline in enumerate(
        portfolio_outline["civilizations"],
        start=1,
    ):
        print("\n" + "#" * 100)

        print(
            "Generating civilization:",
            civilization_number,
            "/",
            CIVILIZATION_COUNT,
        )

        print(
            civilization_outline[
                "civilization_name"
            ]
        )

        evidence_claims, claim_model = (
            generate_evidence_claims(
                civilization_outline,
                brief,
            )
        )

        claim_models[
            civilization_outline["civilization_id"]
        ] = claim_model

        civilization_details, detail_model = (
            generate_civilization_details(
                civilization_outline,
                evidence_claims,
                brief,
            )
        )

        detail_models[
            civilization_outline["civilization_id"]
        ] = detail_model

        combined_civilization_data = {
            **civilization_outline,
            **civilization_details,
            "evidence_claims": [
                claim.model_dump(
                    mode="json"
                )
                for claim in evidence_claims
            ],
        }

        validated_civilization = (
            CivilizationDossier.model_validate(
                combined_civilization_data
            )
        )

        generated_civilizations.append(
            validated_civilization
        )

    portfolio = CivilizationPortfolio(
        exhibition_title=(
            portfolio_outline["exhibition_title"]
        ),
        curator_statement=(
            portfolio_outline["curator_statement"]
        ),
        civilizations=generated_civilizations,
    )

    all_claim_ids = [
        claim.claim_id
        for civilization in portfolio.civilizations
        for claim in civilization.evidence_claims
    ]

    if len(all_claim_ids) != len(
        set(all_claim_ids)
    ):
        raise ValueError(
            "claim_id values must be globally unique."
        )

    model_information = {
        "outline_model": outline_model,
        "claim_models": claim_models,
        "detail_models": detail_models,
    }

    return (
        portfolio,
        model_information,
    )


civilization_portfolio, portfolio_model_id = (
    generate_civilization_portfolio(
        museum_brief
    )
)

structured_portfolio_json = (
    civilization_portfolio.model_dump_json(
        indent=2
    )
)

print("\n" + "=" * 100)
print("CIVILIZATION PORTFOLIO COMPLETED")

print("\nModels used:")
print(
    json.dumps(
        portfolio_model_id,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nStructured portfolio:")
print(structured_portfolio_json)

Operation: Generate civilization portfolio outline
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flash

####################################################################################################
Generating civilization: 1 / 3
The Thalassocracy of Nyxalor
Operation: Generate evidence claims for The Thalassocracy of Nyxalor
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flash
Operation: Generate details for The Thalassocracy of Nyxalor
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flash

####################################################################################################
Generating civilization: 2 / 3
The Zephyrian Astrolatry
Operation: Generate evidence claims for The Zephyrian Astrolatry
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flas

In [21]:
run_id = str(
    uuid4()
)

run_timestamp = datetime.now(
    timezone.utc
)

civilization_rows = []
claim_rows = []

for civilization in civilization_portfolio.civilizations:
    civilization_rows.append(
        {
            "civilization_id": civilization.civilization_id,
            "run_id": run_id,
            "civilization_name": civilization.civilization_name,
            "environment_type": civilization.environment_type,
            "historical_period": civilization.historical_period,
            "civilization_json": civilization.model_dump_json(
                indent=2
            ),
            "created_at": run_timestamp.isoformat(),
        }
    )

    for claim in civilization.evidence_claims:
        claim_rows.append(
            {
                "claim_id": claim.claim_id,
                "run_id": run_id,
                "civilization_id": civilization.civilization_id,
                "claim_type": claim.claim_type,
                "statement": claim.statement,
                "evidence_description": claim.evidence_description,
                "confidence": claim.confidence,
                "permitted_visual_elements": (
                    claim.permitted_visual_elements
                ),
                "forbidden_inferences": (
                    claim.forbidden_inferences
                ),
                "claim_json": claim.model_dump_json(
                    indent=2
                ),
                "created_at": run_timestamp.isoformat(),
            }
        )

print("Run ID:", run_id)
print("Civilizations:", len(civilization_rows))
print("Evidence claims:", len(claim_rows))

display(
    pd.DataFrame(
        [
            {
                "civilization_id": row["civilization_id"],
                "civilization_name": row["civilization_name"],
                "environment_type": row["environment_type"],
                "historical_period": row["historical_period"],
            }
            for row in civilization_rows
        ]
    )
)

display(
    pd.DataFrame(
        [
            {
                "claim_id": row["claim_id"],
                "civilization_id": row["civilization_id"],
                "claim_type": row["claim_type"],
                "confidence": row["confidence"],
                "statement": row["statement"],
            }
            for row in claim_rows
        ]
    )
)

Run ID: a232acc3-9195-4c7f-8513-bb0f4be272fe
Civilizations: 3
Evidence claims: 19


,civilization_id,civilization_name,environment_type,historical_period
0,thalassor_01,The Thalassocracy of Nyxalor,submerged_ocean_city,"Late Bronze Age equivalent, circa 1400 to 1200..."
1,zephyria_02,The Zephyrian Astrolatry,desert_astronomer_city,"Early Iron Age equivalent, circa 800 to 600 BCE"
2,pyroclast_03,The Pyralis Hegemony,geothermal_mountain_city,"Classical Antiquity equivalent, circa 300 to 1..."


,claim_id,civilization_id,claim_type,confidence,statement
0,thalassor_01-claim-01,thalassor_01,artifact,0.85,Weighted bronze diving helmets with integrated...
1,thalassor_01-claim-02,thalassor_01,architecture,0.90,Domed basalt dwellings built directly into the...
2,thalassor_01-claim-03,thalassor_01,infrastructure,0.80,Geothermal heating and ventilation channels ut...
3,thalassor_01-claim-04,thalassor_01,environment,0.75,"Cultivation of giant kelp forests for food, fi..."
4,thalassor_01-claim-05,thalassor_01,social_practice,0.88,Ritualistic burial of the dead in deep volcani...
5,thalassor_01-claim-06,thalassor_01,uncertain_hypothesis,0.35,Domestication of marine mammals for draft work...
6,zephyria_02-claim-01,zephyria_02,artifact,0.92,The Zephyrians manufactured bronze astrolabe-l...
7,zephyria_02-claim-02,zephyria_02,architecture,0.88,Circular sandstone towers with open roofs were...
8,zephyria_02-claim-03,zephyria_02,infrastructure,0.95,"Deep, subterranean cisterns carved into sandst..."
9,zephyria_02-claim-04,zephyria_02,environment,0.97,The region experienced extreme hyper-aridity a...


In [22]:
claim_ledger = {
    "ledger_id": str(uuid4()),
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
    "confidence_threshold_for_generation": (
        CLAIM_CONFIDENCE_THRESHOLD
    ),
    "civilizations": [],
}

for civilization in civilization_portfolio.civilizations:
    accepted_claims = [
        claim.model_dump(
            mode="json"
        )
        for claim in civilization.evidence_claims
        if claim.confidence
        >= CLAIM_CONFIDENCE_THRESHOLD
    ]

    low_confidence_claims = [
        claim.model_dump(
            mode="json"
        )
        for claim in civilization.evidence_claims
        if claim.confidence
        < CLAIM_CONFIDENCE_THRESHOLD
    ]

    claim_ledger["civilizations"].append(
        {
            "civilization_id": civilization.civilization_id,
            "civilization_name": civilization.civilization_name,
            "accepted_for_generation": accepted_claims,
            "excluded_from_generation": low_confidence_claims,
        }
    )

claim_ledger_json = json.dumps(
    claim_ledger,
    indent=2,
    ensure_ascii=False,
    default=str,
)

claim_ledger_blob_name = (
    f"{GCS_CLAIM_PREFIX}/"
    f"{run_id}/"
    "claim_ledger.json"
)

claim_ledger_gcs_uri = upload_text_to_gcs(
    claim_ledger_json,
    blob_name=claim_ledger_blob_name,
    content_type="application/json",
)

print("Claim ledger saved to:")
print(claim_ledger_gcs_uri)

Claim ledger saved to:
gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/claim-ledgers/a232acc3-9195-4c7f-8513-bb0f4be272fe/claim_ledger.json


In [23]:
def build_visual_contract(
    civilization: CivilizationDossier,
) -> str:
    accepted_claims = [
        claim
        for claim in civilization.evidence_claims
        if claim.confidence
        >= CLAIM_CONFIDENCE_THRESHOLD
    ]

    excluded_claims = [
        claim
        for claim in civilization.evidence_claims
        if claim.confidence
        < CLAIM_CONFIDENCE_THRESHOLD
    ]

    permitted_elements = sorted(
        {
            element
            for claim in accepted_claims
            for element in claim.permitted_visual_elements
        }
    )

    forbidden_inferences = sorted(
        {
            inference
            for claim in civilization.evidence_claims
            for inference in claim.forbidden_inferences
        }
    )

    accepted_claim_lines = [
        (
            f"- {claim.claim_id} "
            f"[confidence={claim.confidence:.2f}]: "
            f"{claim.statement}"
        )
        for claim in accepted_claims
    ]

    excluded_claim_lines = [
        (
            f"- {claim.claim_id} "
            f"[confidence={claim.confidence:.2f}]: "
            f"{claim.statement}"
        )
        for claim in excluded_claims
    ]

    return "\n".join(
        [
            f"Civilization ID: {civilization.civilization_id}",
            f"Civilization name: {civilization.civilization_name}",
            "",
            "VISUAL BIBLE:",
            civilization.visual_bible,
            "",
            "ACCEPTED EVIDENCE CLAIMS:",
            *accepted_claim_lines,
            "",
            "LOW-CONFIDENCE CLAIMS EXCLUDED FROM VISUAL GENERATION:",
            *excluded_claim_lines,
            "",
            "PERMITTED VISUAL ELEMENTS:",
            *[
                f"- {element}"
                for element in permitted_elements
            ],
            "",
            "FORBIDDEN INFERENCES:",
            *[
                f"- {inference}"
                for inference in forbidden_inferences
            ],
        ]
    )


visual_contracts = {
    civilization.civilization_id: (
        build_visual_contract(
            civilization
        )
    )
    for civilization in civilization_portfolio.civilizations
}

for civilization_id, contract in visual_contracts.items():
    print("=" * 100)
    print(contract)

Civilization ID: thalassor_01
Civilization name: The Thalassocracy of Nyxalor

VISUAL BIBLE:
The visual aesthetic of Nyxalor is defined by dark, weathered basalt masonry covered in fine marine silt, green algae, and occasional barnacles. Metal elements consist of cast bronze with a heavy, powdery green patina and corroded copper-alloy tubing. Geothermal conduits are characterized by earthy terracotta tones and bright, yellowish-white sulfur encrustations. The environment is dominated by the murky, diffused green-blue light of a temperate ocean shelf, with towering, amber-colored giant kelp fronds swaying in the current. Lighting is naturalistic, filtering down from the surface and occasionally punctuated by the soft, warm glow of hydrothermal vents on the dark seabed.

ACCEPTED EVIDENCE CLAIMS:
- thalassor_01-claim-01 [confidence=0.85]: Weighted bronze diving helmets with integrated copper breathing tubes.
- thalassor_01-claim-02 [confidence=0.90]: Domed basalt dwellings built directly

In [24]:
def extract_image_bytes_from_response(
    response,
) -> tuple[bytes, str]:
    if not response.candidates:
        raise RuntimeError(
            "Image model returned no candidates."
        )

    content = response.candidates[0].content

    if content is None or not content.parts:
        raise RuntimeError(
            "Image model returned no content."
        )

    for part in content.parts:
        inline_data = getattr(
            part,
            "inline_data",
            None,
        )

        if inline_data is None:
            continue

        data = getattr(
            inline_data,
            "data",
            None,
        )

        if not data:
            continue

        if isinstance(data, str):
            image_bytes = base64.b64decode(
                data
            )
        else:
            image_bytes = bytes(
                data
            )

        mime_type = (
            inline_data.mime_type
            or "image/png"
        )

        return (
            image_bytes,
            mime_type,
        )

    raise RuntimeError(
        "No image data found in the response."
    )


def extension_for_image_mime(
    mime_type: str,
) -> str:
    normalized = mime_type.lower()

    if "jpeg" in normalized:
        return "jpg"

    if "webp" in normalized:
        return "webp"

    return "png"


def generate_strict_4k_image(
    *,
    prompt: str,
) -> tuple[
    bytes,
    str,
    str,
    int,
    int,
]:
    errors = []

    for model_id in IMAGE_MODEL_CANDIDATES:
        try:
            print(
                "Trying strict 4K image model:",
                model_id,
            )

            response = call_with_backoff(
                lambda model_id=model_id: (
                    global_genai_client.models.generate_content(
                        model=model_id,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            response_modalities=[
                                types.Modality.TEXT,
                                types.Modality.IMAGE,
                            ],
                            image_config=types.ImageConfig(
                                aspect_ratio=IMAGE_ASPECT_RATIO,
                                image_size=IMAGE_RESOLUTION,
                            ),
                        ),
                    )
                ),
                operation_name=(
                    f"Generate native 4K image "
                    f"with {model_id}"
                ),
            )

            image_bytes, mime_type = (
                extract_image_bytes_from_response(
                    response
                )
            )

            width, height = inspect_image_bytes(
                image_bytes
            )

            assert_native_4k(
                width=width,
                height=height,
                media_label=(
                    f"Image from {model_id}"
                ),
            )

            return (
                image_bytes,
                mime_type,
                model_id,
                width,
                height,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "resolution": IMAGE_RESOLUTION,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Strict 4K image attempt failed:",
                model_id,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "All native 4K image attempts failed. "
        "No lower-resolution fallback or upscaling "
        "will be performed. Errors: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [25]:
def generate_civilization_image(
    civilization: CivilizationDossier,
) -> dict[str, Any]:
    visual_contract = visual_contracts[
        civilization.civilization_id
    ]

    image_prompt = f"""
Create a native 4K photorealistic archaeological reconstruction.

Civilization:
{civilization.civilization_name}

Environment:
{civilization.environment_type}

Historical period:
{civilization.historical_period}

Geographic context:
{civilization.geographic_context}

Architecture:
{civilization.architecture_summary}

Engineering:
{civilization.engineering_summary}

Shared visual contract:
{visual_contract}

Specific image direction:
{civilization.image_prompt}

Global museum visual style:
{museum_brief.visual_style}

Strict requirements:
- native 4K output
- 16:9 landscape composition
- no upscaling
- use only elements permitted by the visual contract
- do not visualize excluded low-confidence claims
- no unsupported technology
- no fantasy magic
- no readable modern text
- no logos
- no copied historical monuments
- no graphic human remains
- premium museum documentary quality
"""

    (
        image_bytes,
        mime_type,
        image_model_id,
        width,
        height,
    ) = generate_strict_4k_image(
        prompt=image_prompt
    )

    extension = extension_for_image_mime(
        mime_type
    )

    blob_name = (
        f"{GCS_IMAGE_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(civilization.civilization_id)}."
        f"{extension}"
    )

    image_gcs_uri = upload_bytes_to_gcs(
        image_bytes,
        blob_name=blob_name,
        content_type=mime_type,
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "civilization_id": civilization.civilization_id,
        "civilization_name": civilization.civilization_name,
        "media_type": "civilization_image",
        "gcs_uri": image_gcs_uri,
        "mime_type": mime_type,
        "generation_model": image_model_id,
        "requested_resolution": IMAGE_RESOLUTION,
        "actual_width": width,
        "actual_height": height,
        "duration_seconds": None,
        "size_bytes": len(image_bytes),
        "native_4k_verified": True,
        "prompt": image_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


image_rows = []

for index, civilization in enumerate(
    civilization_portfolio.civilizations,
    start=1,
):
    print("=" * 100)

    print(
        f"Generating image {index}/{CIVILIZATION_COUNT}:",
        civilization.civilization_name,
    )

    image_row = generate_civilization_image(
        civilization
    )

    image_rows.append(
        image_row
    )

    print(
        "Image:",
        image_row["gcs_uri"],
    )

    print(
        "Verified resolution:",
        image_row["actual_width"],
        "x",
        image_row["actual_height"],
    )

    if index < CIVILIZATION_COUNT:
        time.sleep(
            IMAGE_DELAY_SECONDS
        )

assert len(image_rows) == 3
assert all(
    row["native_4k_verified"]
    for row in image_rows
)

print("Generated native 4K images:", len(image_rows))

Generating image 1/3: The Thalassocracy of Nyxalor
Trying strict 4K image model: gemini-3-pro-image
Image: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/images/a232acc3-9195-4c7f-8513-bb0f4be272fe/thalassor-01.png
Verified resolution: 5504 x 3072
Generating image 2/3: The Zephyrian Astrolatry
Trying strict 4K image model: gemini-3-pro-image
Image: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/images/a232acc3-9195-4c7f-8513-bb0f4be272fe/zephyria-02.png
Verified resolution: 5504 x 3072
Generating image 3/3: The Pyralis Hegemony
Trying strict 4K image model: gemini-3-pro-image
Image: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/images/a232acc3-9195-4c7f-8513-bb0f4be272fe/pyroclast-03.png
Verified resolution: 5504 x 3072
Generated native 4K images: 3


In [ ]:
for image_row in image_rows:
    print("=" * 100)

    print(
        "Civilization:",
        image_row["civilization_name"],
    )

    print(
        "Model:",
        image_row["generation_model"],
    )

    print(
        "Requested resolution:",
        image_row["requested_resolution"],
    )

    print(
        "Actual resolution:",
        image_row["actual_width"],
        "x",
        image_row["actual_height"],
    )

    print(
        "Size:",
        round(
            image_row["size_bytes"]
            / 1_000_000,
            2,
        ),
        "MB",
    )

    print(
        "GCS URI:",
        image_row["gcs_uri"],
    )

    display_gcs_image(
        image_row["gcs_uri"],
        width=760,
    )

In [27]:
MUSEUM_DIALOGUE_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "civilization_id",
        "episode_title",
        "style_prompt",
        "turns",
        "verified_claim_ids",
        "uncertainty_notice",
    ],
    "properties": {
        "civilization_id": {
            "type": "STRING",
        },
        "episode_title": {
            "type": "STRING",
        },
        "style_prompt": {
            "type": "STRING",
        },
        "turns": {
            "type": "ARRAY",
            "minItems": 6,
            "maxItems": 10,
            "items": {
                "type": "OBJECT",
                "required": [
                    "speaker",
                    "text",
                ],
                "properties": {
                    "speaker": {
                        "type": "STRING",
                        "enum": [
                            "Archaeologist",
                            "Curator",
                        ],
                    },
                    "text": {
                        "type": "STRING",
                    },
                },
            },
        },
        "verified_claim_ids": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "uncertainty_notice": {
            "type": "STRING",
        },
    },
}

print("Museum dialogue schema created.")

Museum dialogue schema created.


In [28]:
def generate_museum_dialogue(
    civilization: CivilizationDossier,
) -> tuple[MuseumDialogue, str]:
    allowed_claim_ids = [
        claim.claim_id
        for claim in civilization.evidence_claims
        if claim.confidence
        >= CLAIM_CONFIDENCE_THRESHOLD
    ]

    excluded_claim_ids = [
        claim.claim_id
        for claim in civilization.evidence_claims
        if claim.confidence
        < CLAIM_CONFIDENCE_THRESHOLD
    ]

    prompt = f"""
You are writing a museum audio-guide conversation
between an Archaeologist and a Curator.

Civilization dossier:
{civilization.model_dump_json(indent=2)}

Allowed claim IDs:
{json.dumps(
    allowed_claim_ids,
    indent=2,
)}

Excluded low-confidence claim IDs:
{json.dumps(
    excluded_claim_ids,
    indent=2,
)}

Requirements:
- create between 6 and 8 dialogue turns
- use only speaker aliases Archaeologist and Curator
- explain the most important physical evidence
- distinguish evidence from interpretation
- never present excluded claims as facts
- include an explicit uncertainty notice
- verified_claim_ids must contain only allowed claim IDs
- keep the dialogue suitable for a museum visitor
- use approximately 130 to 220 total spoken words
- returned civilization_id must be exactly:
  {civilization.civilization_id}
- output only valid JSON matching the schema
"""

    raw_dialogue, dialogue_model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=MUSEUM_DIALOGUE_SCHEMA,
            operation_name=(
                f"Generate museum dialogue for "
                f"{civilization.civilization_name}"
            ),
            temperature=0.25,
        )
    )

    dialogue = MuseumDialogue.model_validate(
        raw_dialogue
    )

    if (
        dialogue.civilization_id
        != civilization.civilization_id
    ):
        raise ValueError(
            "Dialogue returned an incorrect civilization_id."
        )

    invalid_claim_ids = (
        set(dialogue.verified_claim_ids)
        - set(allowed_claim_ids)
    )

    if invalid_claim_ids:
        raise ValueError(
            f"Dialogue used unsupported claim IDs: "
            f"{invalid_claim_ids}"
        )

    return (
        dialogue,
        dialogue_model_id,
    )


museum_dialogues = {}
dialogue_models = {}

for civilization in civilization_portfolio.civilizations:
    print("=" * 100)

    print(
        "Generating dialogue:",
        civilization.civilization_name,
    )

    dialogue, dialogue_model_id = (
        generate_museum_dialogue(
            civilization
        )
    )

    museum_dialogues[
        civilization.civilization_id
    ] = dialogue

    dialogue_models[
        civilization.civilization_id
    ] = dialogue_model_id

    print(
        dialogue.model_dump_json(
            indent=2
        )
    )

assert len(museum_dialogues) == 3

Generating dialogue: The Thalassocracy of Nyxalor
Operation: Generate museum dialogue for The Thalassocracy of Nyxalor
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flash
{
  "civilization_id": "thalassor_01",
  "episode_title": "Echoes of the Caldera: The Nyxalor Legacy",
  "style_prompt": "A museum audio guide style soundtrack. Features low, resonant underwater hydrophone recordings, the deep, rhythmic bubbling of geothermal vents, and the distant, muffled creaking of stone structures under ocean pressure. A calm, authoritative narrator speaks with a measured, scholarly tone, accompanied by a minimalist, ambient acoustic soundtrack featuring low-frequency drone notes and sparse, echoing bronze chimes.",
  "turns": [
    {
      "speaker": "Curator",
      "text": "Welcome to the Nyxalor gallery. Here we see the remains of a Late Bronze Age society that lived fifty meters underwater. Archaeologist, how did they survive?"
    },


In [29]:
def dialogue_to_text(
    dialogue: MuseumDialogue,
) -> str:
    return "\n".join(
        f"{turn.speaker}: {turn.text}"
        for turn in dialogue.turns
    )


def generate_multispeaker_audio(
    *,
    dialogue: MuseumDialogue,
    civilization: CivilizationDossier,
) -> tuple[
    bytes,
    str,
    str,
]:
    dialogue_text = dialogue_to_text(
        dialogue
    )

    style_prompt = f"""
{civilization.audio_style_prompt}

{dialogue.style_prompt}

Perform this as a polished museum conversation.

The Archaeologist sounds precise, observant and cautious.
The Curator sounds clear, engaging and visitor-focused.

Do not add music.
Do not add sound effects.
Do not add words that are not present in the dialogue.
Respect the distinction between evidence and speculation.
"""

    errors = []

    for model_id in TTS_MODEL_CANDIDATES:
        try:
            print(
                "Trying TTS model:",
                model_id,
            )

            synthesis_input = texttospeech.SynthesisInput(
                text=dialogue_text,
                prompt=style_prompt,
            )

            multi_speaker_voice_config = (
                texttospeech.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        texttospeech.MultispeakerPrebuiltVoice(
                            speaker_alias="Archaeologist",
                            speaker_id=ARCHAEOLOGIST_VOICE,
                        ),
                        texttospeech.MultispeakerPrebuiltVoice(
                            speaker_alias="Curator",
                            speaker_id=CURATOR_VOICE,
                        ),
                    ]
                )
            )

            voice = texttospeech.VoiceSelectionParams(
                language_code=TTS_LANGUAGE_CODE,
                model_name=model_id,
                multi_speaker_voice_config=(
                    multi_speaker_voice_config
                ),
            )

            audio_config = texttospeech.AudioConfig(
                audio_encoding=(
                    texttospeech.AudioEncoding.LINEAR16
                ),
                sample_rate_hertz=24_000,
            )

            response = call_with_backoff(
                lambda: tts_client.synthesize_speech(
                    input=synthesis_input,
                    voice=voice,
                    audio_config=audio_config,
                ),
                operation_name=(
                    f"Generate museum dialogue "
                    f"with {model_id}"
                ),
            )

            audio_bytes = response.audio_content

            if not audio_bytes:
                raise RuntimeError(
                    "TTS returned empty audio."
                )

            return (
                audio_bytes,
                model_id,
                style_prompt,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "error": str(exc)[:500],
                }
            )

            print(
                "TTS attempt failed:",
                model_id,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "All TTS attempts failed: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [30]:
def generate_civilization_audio(
    civilization: CivilizationDossier,
) -> dict[str, Any]:
    dialogue = museum_dialogues[
        civilization.civilization_id
    ]

    (
        audio_bytes,
        tts_model_id,
        tts_prompt,
    ) = generate_multispeaker_audio(
        dialogue=dialogue,
        civilization=civilization,
    )

    audio_info = inspect_wav_bytes(
        audio_bytes
    )

    blob_name = (
        f"{GCS_AUDIO_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(civilization.civilization_id)}.wav"
    )

    audio_gcs_uri = upload_bytes_to_gcs(
        audio_bytes,
        blob_name=blob_name,
        content_type="audio/wav",
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "civilization_id": civilization.civilization_id,
        "civilization_name": civilization.civilization_name,
        "media_type": "museum_dialogue_audio",
        "gcs_uri": audio_gcs_uri,
        "mime_type": "audio/wav",
        "generation_model": tts_model_id,
        "requested_resolution": None,
        "actual_width": None,
        "actual_height": None,
        "duration_seconds": (
            audio_info["duration_seconds"]
        ),
        "size_bytes": len(audio_bytes),
        "native_4k_verified": None,
        "prompt": tts_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


audio_rows = []

for index, civilization in enumerate(
    civilization_portfolio.civilizations,
    start=1,
):
    print("=" * 100)

    print(
        f"Generating audio {index}/{CIVILIZATION_COUNT}:",
        civilization.civilization_name,
    )

    audio_row = generate_civilization_audio(
        civilization
    )

    audio_rows.append(
        audio_row
    )

    print(
        "Audio:",
        audio_row["gcs_uri"],
    )

    print(
        "Duration:",
        round(
            audio_row["duration_seconds"],
            2,
        ),
        "seconds",
    )

    if index < CIVILIZATION_COUNT:
        time.sleep(
            AUDIO_DELAY_SECONDS
        )

assert len(audio_rows) == 3

print("Generated audio files:", len(audio_rows))

Generating audio 1/3: The Thalassocracy of Nyxalor
Trying TTS model: gemini-3.1-flash-tts-preview
Audio: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/audio/a232acc3-9195-4c7f-8513-bb0f4be272fe/thalassor-01.wav
Duration: 76.84 seconds
Generating audio 2/3: The Zephyrian Astrolatry
Trying TTS model: gemini-3.1-flash-tts-preview
Audio: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/audio/a232acc3-9195-4c7f-8513-bb0f4be272fe/zephyria-02.wav
Duration: 66.36 seconds
Generating audio 3/3: The Pyralis Hegemony
Trying TTS model: gemini-3.1-flash-tts-preview
Audio: gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/audio/a232acc3-9195-4c7f-8513-bb0f4be272fe/pyroclast-03.wav
Duration: 77.72 seconds
Generated audio files: 3


In [31]:
def wait_for_video_operation(
    operation,
    *,
    poll_interval_seconds: int = 20,
    max_wait_seconds: int = 45 * 60,
):
    started_at = time.time()

    while not operation.done:
        elapsed_seconds = int(
            time.time() - started_at
        )

        if elapsed_seconds > max_wait_seconds:
            raise TimeoutError(
                "Video generation exceeded "
                "the maximum waiting time."
            )

        print(
            "Video generation running.",
            "Elapsed seconds:",
            elapsed_seconds,
        )

        time.sleep(
            poll_interval_seconds
        )

        operation = (
            video_genai_client.operations.get(
                operation
            )
        )

    return operation


def extract_video_uri(
    operation,
) -> str:
    operation_error = getattr(
        operation,
        "error",
        None,
    )

    if operation_error:
        raise RuntimeError(
            f"Video operation failed: "
            f"{operation_error}"
        )

    result = getattr(
        operation,
        "result",
        None,
    )

    if result is None:
        result = getattr(
            operation,
            "response",
            None,
        )

    if result is None:
        raise RuntimeError(
            "Video operation returned no result."
        )

    generated_videos = getattr(
        result,
        "generated_videos",
        None,
    )

    if not generated_videos:
        raise RuntimeError(
            "No generated videos were returned."
        )

    video_object = generated_videos[0].video

    video_uri = (
        getattr(
            video_object,
            "uri",
            None,
        )
        or getattr(
            video_object,
            "gcs_uri",
            None,
        )
    )

    if not video_uri:
        raise RuntimeError(
            "Could not extract the generated video URI."
        )

    return video_uri

In [32]:
def generate_strict_4k_video(
    *,
    prompt: str,
    output_prefix: str,
) -> tuple[
    str,
    dict[str, Any],
]:
    print(
        "Generating strict native 4K video with:",
        VIDEO_MODEL,
    )

    operation = call_with_backoff(
        lambda: video_genai_client.models.generate_videos(
            model=VIDEO_MODEL,
            prompt=prompt,
            config=types.GenerateVideosConfig(
                number_of_videos=1,
                duration_seconds=(
                    VIDEO_DURATION_SECONDS
                ),
                aspect_ratio=VIDEO_ASPECT_RATIO,
                resolution=VIDEO_RESOLUTION,
                output_gcs_uri=output_prefix,
            ),
        ),
        operation_name=(
            f"Start native 4K video with {VIDEO_MODEL}"
        ),
    )

    completed_operation = wait_for_video_operation(
        operation
    )

    video_gcs_uri = extract_video_uri(
        completed_operation
    )

    video_bytes = download_gcs_bytes(
        video_gcs_uri
    )

    video_info = inspect_video_bytes(
        video_bytes
    )

    assert_native_4k(
        width=video_info["width"],
        height=video_info["height"],
        media_label=(
            f"Video from {VIDEO_MODEL}"
        ),
    )

    return (
        video_gcs_uri,
        {
            **video_info,
            "size_bytes": len(video_bytes),
        },
    )

In [33]:
def generate_civilization_video(
    civilization: CivilizationDossier,
) -> dict[str, Any]:
    visual_contract = visual_contracts[
        civilization.civilization_id
    ]

    video_prompt = f"""
Create one continuous native 4K archaeological documentary shot.

Civilization:
{civilization.civilization_name}

Environment:
{civilization.environment_type}

Historical period:
{civilization.historical_period}

Geographic context:
{civilization.geographic_context}

Architecture:
{civilization.architecture_summary}

Engineering:
{civilization.engineering_summary}

Daily life:
{civilization.daily_life_summary}

Shared visual contract:
{visual_contract}

Specific motion direction:
{civilization.video_prompt}

Global museum style:
{museum_brief.visual_style}

Strict requirements:
- native 4K output
- 16:9 aspect ratio
- eight-second duration
- one continuous shot
- no cuts
- use only elements permitted by the visual contract
- do not visualize excluded low-confidence claims
- no unsupported technology
- no copied historical monuments
- no fantasy magic
- no readable modern text
- no logos
- no morphing architecture
- no disappearing structures
- no duplicated people
- no dialogue
- premium archaeological documentary quality
"""

    output_prefix = (
        f"gs://{BUCKET_NAME}/"
        f"{GCS_VIDEO_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(civilization.civilization_id)}/"
    )

    (
        video_gcs_uri,
        video_info,
    ) = generate_strict_4k_video(
        prompt=video_prompt,
        output_prefix=output_prefix,
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "civilization_id": civilization.civilization_id,
        "civilization_name": civilization.civilization_name,
        "media_type": "civilization_video",
        "gcs_uri": video_gcs_uri,
        "mime_type": "video/mp4",
        "generation_model": VIDEO_MODEL,
        "requested_resolution": VIDEO_RESOLUTION,
        "actual_width": video_info["width"],
        "actual_height": video_info["height"],
        "duration_seconds": (
            video_info["duration_seconds"]
        ),
        "size_bytes": video_info["size_bytes"],
        "native_4k_verified": True,
        "prompt": video_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


video_rows = []

for index, civilization in enumerate(
    civilization_portfolio.civilizations,
    start=1,
):
    print("=" * 100)

    print(
        f"Generating video {index}/{CIVILIZATION_COUNT}:",
        civilization.civilization_name,
    )

    video_row = generate_civilization_video(
        civilization
    )

    video_rows.append(
        video_row
    )

    print(
        "Video:",
        video_row["gcs_uri"],
    )

    print(
        "Verified resolution:",
        video_row["actual_width"],
        "x",
        video_row["actual_height"],
    )

    if index < CIVILIZATION_COUNT:
        time.sleep(
            VIDEO_DELAY_SECONDS
        )

assert len(video_rows) == 3
assert all(
    row["native_4k_verified"]
    for row in video_rows
)

print("Generated native 4K videos:", len(video_rows))

Generating video 1/3: The Thalassocracy of Nyxalor
Generating strict native 4K video with: veo-3.1-generate-001
Video generation running. Elapsed seconds: 0
Video generation running. Elapsed seconds: 20
Video generation running. Elapsed seconds: 41
Video generation running. Elapsed seconds: 62
Video generation running. Elapsed seconds: 83
Video generation running. Elapsed seconds: 104
Video generation running. Elapsed seconds: 125
Video generation running. Elapsed seconds: 146
Video generation running. Elapsed seconds: 167
Video generation running. Elapsed seconds: 188
Video generation running. Elapsed seconds: 209
Video generation running. Elapsed seconds: 230
Video generation running. Elapsed seconds: 251
Video generation running. Elapsed seconds: 272
Video generation running. Elapsed seconds: 293
Video generation running. Elapsed seconds: 314
Video generation running. Elapsed seconds: 335
Video generation running. Elapsed seconds: 356
Video generation running. Elapsed seconds: 377
V

In [34]:
MEDIA_VERIFICATION_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "civilization_id",
        "image_summary",
        "video_summary",
        "audio_summary",
        "supported_claim_ids",
        "unsupported_visual_elements",
        "contradictory_elements",
        "image_claim_fidelity_score",
        "video_claim_fidelity_score",
        "audio_evidence_fidelity_score",
        "cross_modal_consistency_score",
        "historical_plausibility_score",
        "hallucination_risk",
        "recommended_corrections",
    ],
    "properties": {
        "civilization_id": {
            "type": "STRING",
        },
        "image_summary": {
            "type": "STRING",
        },
        "video_summary": {
            "type": "STRING",
        },
        "audio_summary": {
            "type": "STRING",
        },
        "supported_claim_ids": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "unsupported_visual_elements": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "contradictory_elements": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "image_claim_fidelity_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "video_claim_fidelity_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "audio_evidence_fidelity_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "cross_modal_consistency_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "historical_plausibility_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "hallucination_risk": {
            "type": "STRING",
            "enum": [
                "low",
                "medium",
                "high",
            ],
        },
        "recommended_corrections": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
    },
}

print("Media verification schema created.")

Media verification schema created.


In [35]:
def find_civilization(
    civilization_id: str,
) -> CivilizationDossier:
    for civilization in civilization_portfolio.civilizations:
        if (
            civilization.civilization_id
            == civilization_id
        ):
            return civilization

    raise KeyError(
        f"Unknown civilization_id: {civilization_id}"
    )


def verify_civilization_media(
    civilization: CivilizationDossier,
    image_row: dict[str, Any],
    audio_row: dict[str, Any],
    video_row: dict[str, Any],
) -> tuple[
    MediaVerification,
    str,
]:
    dialogue = museum_dialogues[
        civilization.civilization_id
    ]

    allowed_claim_ids = [
        claim.claim_id
        for claim in civilization.evidence_claims
        if claim.confidence
        >= CLAIM_CONFIDENCE_THRESHOLD
    ]

    prompt = f"""
You are an archaeological evidence auditor,
museum ethics reviewer and multimodal quality supervisor.

Evaluate the image, video and audio against the claim ledger.

Civilization dossier:
{civilization.model_dump_json(indent=2)}

Visual contract:
{visual_contracts[civilization.civilization_id]}

Structured dialogue:
{dialogue.model_dump_json(indent=2)}

Allowed claim IDs:
{json.dumps(
    allowed_claim_ids,
    indent=2,
)}

Tasks:
- summarize what is visible in the image
- summarize what happens in the video
- summarize what the audio claims
- identify which claims are visibly or verbally supported
- find visual elements not supported by the claim ledger
- detect contradictions between image and video
- detect claims presented too confidently in the audio
- evaluate historical plausibility
- evaluate cross-modal consistency
- assign hallucination risk
- returned civilization_id must be exactly:
  {civilization.civilization_id}

Return only valid JSON matching the schema.
"""

    raw_verification, verification_model_id = (
        generate_structured_with_fallback(
            contents=[
                types.Part.from_uri(
                    file_uri=image_row["gcs_uri"],
                    mime_type=image_row["mime_type"],
                ),
                types.Part.from_uri(
                    file_uri=audio_row["gcs_uri"],
                    mime_type=audio_row["mime_type"],
                ),
                types.Part.from_uri(
                    file_uri=video_row["gcs_uri"],
                    mime_type=video_row["mime_type"],
                ),
                prompt,
            ],
            response_schema=MEDIA_VERIFICATION_SCHEMA,
            operation_name=(
                f"Verify multimedia for "
                f"{civilization.civilization_name}"
            ),
            temperature=0.1,
        )
    )

    verification = MediaVerification.model_validate(
        raw_verification
    )

    if (
        verification.civilization_id
        != civilization.civilization_id
    ):
        raise ValueError(
            "Verification returned an incorrect civilization_id."
        )

    invalid_claim_ids = (
        set(verification.supported_claim_ids)
        - set(allowed_claim_ids)
    )

    if invalid_claim_ids:
        raise ValueError(
            f"Verification returned unsupported claim IDs: "
            f"{invalid_claim_ids}"
        )

    return (
        verification,
        verification_model_id,
    )


media_verifications = {}
review_rows = []

for civilization in civilization_portfolio.civilizations:
    image_row = next(
        row
        for row in image_rows
        if (
            row["civilization_id"]
            == civilization.civilization_id
        )
    )

    audio_row = next(
        row
        for row in audio_rows
        if (
            row["civilization_id"]
            == civilization.civilization_id
        )
    )

    video_row = next(
        row
        for row in video_rows
        if (
            row["civilization_id"]
            == civilization.civilization_id
        )
    )

    print("=" * 100)

    print(
        "Verifying:",
        civilization.civilization_name,
    )

    verification, verification_model_id = (
        verify_civilization_media(
            civilization,
            image_row,
            audio_row,
            video_row,
        )
    )

    media_verifications[
        civilization.civilization_id
    ] = verification

    review_rows.append(
        {
            "review_id": str(uuid4()),
            "run_id": run_id,
            "civilization_id": (
                civilization.civilization_id
            ),
            "review_json": (
                verification.model_dump_json(
                    indent=2
                )
            ),
            "review_model": verification_model_id,
            "created_at": (
                datetime.now(timezone.utc).isoformat()
            ),
        }
    )

    print(
        verification.model_dump_json(
            indent=2
        )
    )

assert len(media_verifications) == 3

Verifying: The Thalassocracy of Nyxalor
Operation: Verify multimedia for The Thalassocracy of Nyxalor
Trying reasoning model: gemini-3.5-flash
Structured response generated successfully.
Model: gemini-3.5-flash
{
  "civilization_id": "thalassor_01",
  "image_summary": "A high-resolution archaeological reconstruction of a submerged Nyxalor dwelling. A circular, single-story dome made of interlocking dark basalt blocks sits nestled against a volcanic caldera wall under fifty meters of green-blue ocean water. Dappled sunlight filters down through a dense canopy of giant kelp. In the foreground, a weathered terracotta pipe with yellow sulfur crusts runs along a stone trench. Resting on the seabed near the sliding stone door of the dwelling is a discarded, heavy cast bronze diving helmet with a green patina, a circular mica viewing port, and a corroded copper tube.",
  "video_summary": "An eight-second continuous archaeological documentary camera shot. The camera slowly glides forward throu

In [36]:
document_rows = []


def add_document(
    *,
    document_type: str,
    title: str,
    content: str,
    civilization_id: str | None = None,
) -> None:
    document_rows.append(
        {
            "document_type": document_type,
            "title": title,
            "content": content,
            "civilization_id": civilization_id,
        }
    )


add_document(
    document_type="museum_brief",
    title="Lost Civilizations Museum Brief",
    content=structured_brief_json,
)

add_document(
    document_type="civilization_portfolio",
    title=civilization_portfolio.exhibition_title,
    content=structured_portfolio_json,
)

add_document(
    document_type="claim_ledger",
    title="Archaeological Claim Ledger",
    content=claim_ledger_json,
)

for civilization in civilization_portfolio.civilizations:
    civilization_id = civilization.civilization_id

    add_document(
        document_type="civilization_dossier",
        title=civilization.civilization_name,
        civilization_id=civilization_id,
        content=civilization.model_dump_json(
            indent=2
        ),
    )

    add_document(
        document_type="visual_contract",
        title=(
            f"Visual Contract - "
            f"{civilization.civilization_name}"
        ),
        civilization_id=civilization_id,
        content=visual_contracts[
            civilization_id
        ],
    )

    dialogue = museum_dialogues[
        civilization_id
    ]

    add_document(
        document_type="museum_dialogue",
        title=dialogue.episode_title,
        civilization_id=civilization_id,
        content=dialogue.model_dump_json(
            indent=2
        ),
    )

    verification = media_verifications[
        civilization_id
    ]

    add_document(
        document_type="media_verification",
        title=(
            f"Media Verification - "
            f"{civilization.civilization_name}"
        ),
        civilization_id=civilization_id,
        content=verification.model_dump_json(
            indent=2
        ),
    )

for media_row in [
    *image_rows,
    *audio_rows,
    *video_rows,
]:
    media_content = "\n".join(
        [
            (
                f"Civilization ID: "
                f"{media_row['civilization_id']}"
            ),
            (
                f"Civilization: "
                f"{media_row['civilization_name']}"
            ),
            (
                f"Media type: "
                f"{media_row['media_type']}"
            ),
            (
                f"Generation model: "
                f"{media_row['generation_model']}"
            ),
            (
                f"Requested resolution: "
                f"{media_row['requested_resolution']}"
            ),
            (
                f"Actual dimensions: "
                f"{media_row['actual_width']}x"
                f"{media_row['actual_height']}"
            ),
            (
                f"Native 4K verified: "
                f"{media_row['native_4k_verified']}"
            ),
            (
                f"GCS URI: "
                f"{media_row['gcs_uri']}"
            ),
            (
                f"Prompt: "
                f"{media_row['prompt']}"
            ),
        ]
    )

    add_document(
        document_type="media_asset_card",
        title=(
            f"{media_row['media_type']} - "
            f"{media_row['civilization_name']}"
        ),
        civilization_id=(
            media_row["civilization_id"]
        ),
        content=media_content,
    )

print("Knowledge documents:", len(document_rows))

display(
    pd.DataFrame(
        [
            {
                "document_type": row["document_type"],
                "civilization_id": row["civilization_id"],
                "title": row["title"],
                "content_length": len(row["content"]),
            }
            for row in document_rows
        ]
    )
)

Knowledge documents: 24


,document_type,civilization_id,title,content_length
0,museum_brief,None,Lost Civilizations Museum Brief,1634
1,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,36451
2,claim_ledger,None,Archaeological Claim Ledger,17722
3,civilization_dossier,thalassor_01,The Thalassocracy of Nyxalor,11920
4,visual_contract,thalassor_01,Visual Contract - The Thalassocracy of Nyxalor,3272
5,museum_dialogue,thalassor_01,Echoes of the Caldera: The Nyxalor Legacy,2562
6,media_verification,thalassor_01,Media Verification - The Thalassocracy of Nyxalor,2131
7,civilization_dossier,zephyria_02,The Zephyrian Astrolatry,10839
8,visual_contract,zephyria_02,Visual Contract - The Zephyrian Astrolatry,2947
9,museum_dialogue,zephyria_02,Echoes of the Desert Sky,2175


In [37]:
document_rows = []


def add_document(
    *,
    document_type: str,
    title: str,
    content: str,
    civilization_id: str | None = None,
) -> None:
    document_rows.append(
        {
            "document_type": document_type,
            "title": title,
            "content": content,
            "civilization_id": civilization_id,
        }
    )


add_document(
    document_type="museum_brief",
    title="Lost Civilizations Museum Brief",
    content=structured_brief_json,
)

add_document(
    document_type="civilization_portfolio",
    title=civilization_portfolio.exhibition_title,
    content=structured_portfolio_json,
)

add_document(
    document_type="claim_ledger",
    title="Archaeological Claim Ledger",
    content=claim_ledger_json,
)

for civilization in civilization_portfolio.civilizations:
    civilization_id = civilization.civilization_id

    add_document(
        document_type="civilization_dossier",
        title=civilization.civilization_name,
        civilization_id=civilization_id,
        content=civilization.model_dump_json(
            indent=2
        ),
    )

    add_document(
        document_type="visual_contract",
        title=(
            f"Visual Contract - "
            f"{civilization.civilization_name}"
        ),
        civilization_id=civilization_id,
        content=visual_contracts[
            civilization_id
        ],
    )

    dialogue = museum_dialogues[
        civilization_id
    ]

    add_document(
        document_type="museum_dialogue",
        title=dialogue.episode_title,
        civilization_id=civilization_id,
        content=dialogue.model_dump_json(
            indent=2
        ),
    )

    verification = media_verifications[
        civilization_id
    ]

    add_document(
        document_type="media_verification",
        title=(
            f"Media Verification - "
            f"{civilization.civilization_name}"
        ),
        civilization_id=civilization_id,
        content=verification.model_dump_json(
            indent=2
        ),
    )

for media_row in [
    *image_rows,
    *audio_rows,
    *video_rows,
]:
    media_content = "\n".join(
        [
            (
                f"Civilization ID: "
                f"{media_row['civilization_id']}"
            ),
            (
                f"Civilization: "
                f"{media_row['civilization_name']}"
            ),
            (
                f"Media type: "
                f"{media_row['media_type']}"
            ),
            (
                f"Generation model: "
                f"{media_row['generation_model']}"
            ),
            (
                f"Requested resolution: "
                f"{media_row['requested_resolution']}"
            ),
            (
                f"Actual dimensions: "
                f"{media_row['actual_width']}x"
                f"{media_row['actual_height']}"
            ),
            (
                f"Native 4K verified: "
                f"{media_row['native_4k_verified']}"
            ),
            (
                f"GCS URI: "
                f"{media_row['gcs_uri']}"
            ),
            (
                f"Prompt: "
                f"{media_row['prompt']}"
            ),
        ]
    )

    add_document(
        document_type="media_asset_card",
        title=(
            f"{media_row['media_type']} - "
            f"{media_row['civilization_name']}"
        ),
        civilization_id=(
            media_row["civilization_id"]
        ),
        content=media_content,
    )

print("Knowledge documents:", len(document_rows))

display(
    pd.DataFrame(
        [
            {
                "document_type": row["document_type"],
                "civilization_id": row["civilization_id"],
                "title": row["title"],
                "content_length": len(row["content"]),
            }
            for row in document_rows
        ]
    )
)

Knowledge documents: 24


,document_type,civilization_id,title,content_length
0,museum_brief,None,Lost Civilizations Museum Brief,1634
1,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,36451
2,claim_ledger,None,Archaeological Claim Ledger,17722
3,civilization_dossier,thalassor_01,The Thalassocracy of Nyxalor,11920
4,visual_contract,thalassor_01,Visual Contract - The Thalassocracy of Nyxalor,3272
5,museum_dialogue,thalassor_01,Echoes of the Caldera: The Nyxalor Legacy,2562
6,media_verification,thalassor_01,Media Verification - The Thalassocracy of Nyxalor,2131
7,civilization_dossier,zephyria_02,The Zephyrian Astrolatry,10839
8,visual_contract,zephyria_02,Visual Contract - The Zephyrian Astrolatry,2947
9,museum_dialogue,zephyria_02,Echoes of the Desert Sky,2175


In [38]:
def normalize_text(
    text: str,
) -> str:
    return " ".join(
        text.split()
    )


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(
        text
    )

    if len(clean_text) <= chunk_size_chars:
        return [
            clean_text
        ]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(
            start + chunk_size_chars,
            len(clean_text),
        )

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(
                ". ",
                start,
                end,
            )

            if sentence_boundary > (
                start
                + int(chunk_size_chars * 0.6)
            ):
                end = sentence_boundary + 1

        chunk = clean_text[
            start:end
        ].strip()

        if chunk:
            chunks.append(
                chunk
            )

        if end >= len(clean_text):
            break

        start = max(
            0,
            end - overlap_chars,
        )

    return chunks

In [39]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for document in document_rows:
    chunks = split_text_into_chunks(
        document["content"]
    )

    for chunk_number, chunk_text in enumerate(
        chunks,
        start=1,
    ):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "run_id": run_id,
                "civilization_id": (
                    document["civilization_id"]
                ),
                "document_type": (
                    document["document_type"]
                ),
                "title": document["title"],
                "chunk_number": chunk_number,
                "global_chunk_number": (
                    global_chunk_number
                ),
                "chunk_text": chunk_text,
                "embedding_model": EMBEDDING_MODEL,
                "embedding_dimension": (
                    EMBEDDING_DIMENSION
                ),
                "created_at": (
                    datetime.now(
                        timezone.utc
                    ).isoformat()
                ),
            }
        )

        global_chunk_number += 1

print(
    "Chunks without embeddings:",
    len(chunk_rows_without_embeddings),
)

display(
    pd.DataFrame(
        [
            {
                "global_chunk_number": (
                    row["global_chunk_number"]
                ),
                "document_type": (
                    row["document_type"]
                ),
                "civilization_id": (
                    row["civilization_id"]
                ),
                "title": row["title"],
                "preview": (
                    row["chunk_text"][:180]
                ),
            }
            for row in chunk_rows_without_embeddings
        ]
    ).head(25)
)

Chunks without embeddings: 182


,global_chunk_number,document_type,civilization_id,title,preview
0,1,museum_brief,None,Lost Civilizations Museum Brief,"{ ""exhibition_title"": ""Echoes Beneath Stone: T..."
1,2,museum_brief,None,Lost Civilizations Museum Brief,ospheric natural light and high-detail environ...
2,3,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,"{ ""exhibition_title"": ""Echoes Beneath Stone: T..."
3,4,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,"sts."", ""discovery_story"": ""The ruins of Nyxalo..."
4,5,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,dwellings constructed from interlocking blocks...
5,6,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,asalt floors of the residential sectors. Flow ...
6,7,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,"ld stone and bronze sickles, processing the ma..."
7,8,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,"lowered the caldera floor by several meters, s..."
8,9,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,crustations. The environment is dominated by t...
9,10,civilization_portfolio,None,Echoes Beneath Stone: Three Civilizations That...,"metal collar rings"" ], ""forbidden_inferences"":..."


In [40]:
def generate_text_embedding(
    text: str,
    *,
    task_type: str,
) -> list[float]:
    response = call_with_backoff(
        lambda: global_genai_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text,
            config=types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=(
                    EMBEDDING_DIMENSION
                ),
            ),
        ),
        operation_name=(
            f"Generate {task_type} embedding"
        ),
    )

    if not response.embeddings:
        raise RuntimeError(
            "Embedding model returned no embeddings."
        )

    embedding = list(
        response.embeddings[0].values
    )

    if len(embedding) != EMBEDDING_DIMENSION:
        raise ValueError(
            f"Expected dimension "
            f"{EMBEDDING_DIMENSION}, "
            f"got {len(embedding)}."
        )

    return embedding


chunk_rows = []

embedding_started_at = time.perf_counter()

for index, row in enumerate(
    chunk_rows_without_embeddings,
    start=1,
):
    print(
        f"Embedding chunk "
        f"{index}/"
        f"{len(chunk_rows_without_embeddings)}"
    )

    embedding = generate_text_embedding(
        row["chunk_text"],
        task_type="RETRIEVAL_DOCUMENT",
    )

    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

    if (
        index
        < len(chunk_rows_without_embeddings)
    ):
        time.sleep(
            EMBEDDING_DELAY_SECONDS
        )

embedding_seconds = (
    time.perf_counter()
    - embedding_started_at
)

print("Chunks with embeddings:", len(chunk_rows))
print(
    "Embedding seconds:",
    round(
        embedding_seconds,
        2,
    ),
)
print(
    "Embedding dimension:",
    len(chunk_rows[0]["embedding"]),
)

Embedding chunk 1/182
Embedding chunk 2/182
Embedding chunk 3/182
Embedding chunk 4/182
Embedding chunk 5/182
Embedding chunk 6/182
Embedding chunk 7/182
Embedding chunk 8/182
Embedding chunk 9/182
Embedding chunk 10/182
Embedding chunk 11/182
Embedding chunk 12/182
Embedding chunk 13/182
Embedding chunk 14/182
Embedding chunk 15/182
Embedding chunk 16/182
Embedding chunk 17/182
Embedding chunk 18/182
Embedding chunk 19/182
Embedding chunk 20/182
Embedding chunk 21/182
Embedding chunk 22/182
Embedding chunk 23/182
Embedding chunk 24/182
Embedding chunk 25/182
Embedding chunk 26/182
Embedding chunk 27/182
Embedding chunk 28/182
Embedding chunk 29/182
Embedding chunk 30/182
Embedding chunk 31/182
Embedding chunk 32/182
Embedding chunk 33/182
Embedding chunk 34/182
Embedding chunk 35/182
Embedding chunk 36/182
Embedding chunk 37/182
Embedding chunk 38/182
Embedding chunk 39/182
Embedding chunk 40/182
Embedding chunk 41/182
Embedding chunk 42/182
Embedding chunk 43/182
Embedding chunk 44/1

In [41]:
all_media_rows = [
    *image_rows,
    *audio_rows,
    *video_rows,
]

assert len(image_rows) == 3
assert len(audio_rows) == 3
assert len(video_rows) == 3
assert len(all_media_rows) == 9

assert all(
    row["native_4k_verified"]
    for row in image_rows
)

assert all(
    row["native_4k_verified"]
    for row in video_rows
)

model_selection = {
    "portfolio_model": portfolio_model_id,
    "dialogue_models": dialogue_models,
    "image_models_used": sorted(
        {
            row["generation_model"]
            for row in image_rows
        }
    ),
    "image_resolution": IMAGE_RESOLUTION,
    "tts_models_used": sorted(
        {
            row["generation_model"]
            for row in audio_rows
        }
    ),
    "video_model": VIDEO_MODEL,
    "video_resolution": VIDEO_RESOLUTION,
    "verification_models": sorted(
        {
            row["review_model"]
            for row in review_rows
        }
    ),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "upscaling_used": False,
    "downscaling_used": False,
}

run_row = {
    "run_id": run_id,
    "exhibition_title": (
        civilization_portfolio.exhibition_title
    ),
    "structured_brief_json": structured_brief_json,
    "structured_portfolio_json": (
        structured_portfolio_json
    ),
    "claim_ledger_gcs_uri": (
        claim_ledger_gcs_uri
    ),
    "model_selection_json": json.dumps(
        model_selection,
        indent=2,
        ensure_ascii=False,
    ),
    "created_at": run_timestamp.isoformat(),
}

print(
    json.dumps(
        model_selection,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "portfolio_model": {
    "outline_model": "gemini-3.5-flash",
    "claim_models": {
      "thalassor_01": "gemini-3.5-flash",
      "zephyria_02": "gemini-3.5-flash",
      "pyroclast_03": "gemini-3.5-flash"
    },
    "detail_models": {
      "thalassor_01": "gemini-3.5-flash",
      "zephyria_02": "gemini-3.5-flash",
      "pyroclast_03": "gemini-3.5-flash"
    }
  },
  "dialogue_models": {
    "thalassor_01": "gemini-3.5-flash",
    "zephyria_02": "gemini-3.5-flash",
    "pyroclast_03": "gemini-3.5-flash"
  },
  "image_models_used": [
    "gemini-3-pro-image"
  ],
  "image_resolution": "4K",
  "tts_models_used": [
    "gemini-3.1-flash-tts-preview"
  ],
  "video_model": "veo-3.1-generate-001",
  "video_resolution": "4k",
  "verification_models": [
    "gemini-3.5-flash"
  ],
  "embedding_model": "gemini-embedding-001",
  "embedding_dimension": 1536,
  "upscaling_used": false,
  "downscaling_used": false
}


In [42]:
run_load_result = batch_load_rows_to_bigquery(
    rows=[run_row],
    table_ref=run_table_ref,
    schema=run_schema,
    table_name=RUN_TABLE_ID,
    run_id=run_id,
)

civilization_load_result = (
    batch_load_rows_to_bigquery(
        rows=civilization_rows,
        table_ref=civilization_table_ref,
        schema=civilization_schema,
        table_name=CIVILIZATION_TABLE_ID,
        run_id=run_id,
    )
)

claim_load_result = batch_load_rows_to_bigquery(
    rows=claim_rows,
    table_ref=claim_table_ref,
    schema=claim_schema,
    table_name=CLAIM_TABLE_ID,
    run_id=run_id,
)

media_load_result = batch_load_rows_to_bigquery(
    rows=all_media_rows,
    table_ref=media_table_ref,
    schema=media_schema,
    table_name=MEDIA_TABLE_ID,
    run_id=run_id,
)

review_load_result = batch_load_rows_to_bigquery(
    rows=review_rows,
    table_ref=review_table_ref,
    schema=review_schema,
    table_name=REVIEW_TABLE_ID,
    run_id=run_id,
)

chunk_load_result = batch_load_rows_to_bigquery(
    rows=chunk_rows,
    table_ref=chunk_table_ref,
    schema=chunk_schema,
    table_name=CHUNK_TABLE_ID,
    run_id=run_id,
)

print(run_load_result)
print(civilization_load_result)
print(claim_load_result)
print(media_load_result)
print(review_load_result)
print(chunk_load_result)

{'table_name': 'museum_runs', 'rows_loaded': 1, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/museum_runs.ndjson', 'job_id': '118f987d-f7ff-4e02-b711-93d1ab73bcca', 'elapsed_seconds': 2.783}
{'table_name': 'civilizations', 'rows_loaded': 3, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/civilizations.ndjson', 'job_id': '0270d78f-b279-4f7c-934d-fc0181c0e594', 'elapsed_seconds': 2.865}
{'table_name': 'evidence_claims', 'rows_loaded': 19, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/evidence_claims.ndjson', 'job_id': '00a302cd-e29e-4a0d-8478-0b800fd2aea5', 'elapsed_seconds': 1.97}
{'table_name': 'museum_media', 'rows_loaded': 9, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/museum_m

In [43]:
sql = f"""
SELECT
  media_type,
  generation_model,
  requested_resolution,
  actual_width,
  actual_height,
  native_4k_verified,
  COUNT(*) AS asset_count,
  SUM(size_bytes) AS total_bytes
FROM `{media_table_ref}`
WHERE run_id = @run_id
GROUP BY
  media_type,
  generation_model,
  requested_resolution,
  actual_width,
  actual_height,
  native_4k_verified
ORDER BY
  media_type,
  generation_model
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            "run_id",
            "STRING",
            run_id,
        )
    ]
)

media_verification_df = (
    bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )
    .to_dataframe()
)

display(
    media_verification_df
)

media_counts = (
    media_verification_df
    .groupby("media_type")["asset_count"]
    .sum()
    .to_dict()
)

assert media_counts["civilization_image"] == 3
assert media_counts["museum_dialogue_audio"] == 3
assert media_counts["civilization_video"] == 3

visual_media_df = media_verification_df[
    media_verification_df["media_type"].isin(
        [
            "civilization_image",
            "civilization_video",
        ]
    )
]

assert visual_media_df[
    "native_4k_verified"
].all()

assert (
    visual_media_df["actual_width"]
    >= MIN_4K_WIDTH
).all()

assert (
    visual_media_df["actual_height"]
    >= MIN_4K_HEIGHT
).all()

print("Validation passed.")
print("- 3 native 4K images")
print("- 3 audio files")
print("- 3 native 4K videos")
print("- no upscaling")
print("- no downscaling")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,media_type,generation_model,requested_resolution,actual_width,actual_height,native_4k_verified,asset_count,total_bytes
0,civilization_image,gemini-3-pro-image,4K,5504,3072,True,3,73193901
1,civilization_video,veo-3.1-generate-001,4k,3840,2160,True,3,299808387
2,museum_dialogue_audio,gemini-3.1-flash-tts-preview,None,<NA>,<NA>,<NA>,3,10604292


Validation passed.
- 3 native 4K images
- 3 audio files
- 3 native 4K videos
- no upscaling
- no downscaling


In [44]:
sql = f"""
SELECT
  media_type,
  generation_model,
  requested_resolution,
  actual_width,
  actual_height,
  native_4k_verified,
  COUNT(*) AS asset_count,
  SUM(size_bytes) AS total_bytes
FROM `{media_table_ref}`
WHERE run_id = @run_id
GROUP BY
  media_type,
  generation_model,
  requested_resolution,
  actual_width,
  actual_height,
  native_4k_verified
ORDER BY
  media_type,
  generation_model
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            "run_id",
            "STRING",
            run_id,
        )
    ]
)

media_verification_df = (
    bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )
    .to_dataframe()
)

display(
    media_verification_df
)

media_counts = (
    media_verification_df
    .groupby("media_type")["asset_count"]
    .sum()
    .to_dict()
)

assert media_counts["civilization_image"] == 3
assert media_counts["museum_dialogue_audio"] == 3
assert media_counts["civilization_video"] == 3

visual_media_df = media_verification_df[
    media_verification_df["media_type"].isin(
        [
            "civilization_image",
            "civilization_video",
        ]
    )
]

assert visual_media_df[
    "native_4k_verified"
].all()

assert (
    visual_media_df["actual_width"]
    >= MIN_4K_WIDTH
).all()

assert (
    visual_media_df["actual_height"]
    >= MIN_4K_HEIGHT
).all()

print("Validation passed.")
print("- 3 native 4K images")
print("- 3 audio files")
print("- 3 native 4K videos")
print("- no upscaling")
print("- no downscaling")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,media_type,generation_model,requested_resolution,actual_width,actual_height,native_4k_verified,asset_count,total_bytes
0,civilization_image,gemini-3-pro-image,4K,5504,3072,True,3,73193901
1,civilization_video,veo-3.1-generate-001,4k,3840,2160,True,3,299808387
2,museum_dialogue_audio,gemini-3.1-flash-tts-preview,None,<NA>,<NA>,<NA>,3,10604292


Validation passed.
- 3 native 4K images
- 3 audio files
- 3 native 4K videos
- no upscaling
- no downscaling


In [45]:
QUERY_EMBEDDING_CACHE: dict[
    str,
    list[float],
] = {}


def get_query_embedding(
    query: str,
) -> list[float]:
    clean_query = normalize_text(
        query
    )

    cached_embedding = (
        QUERY_EMBEDDING_CACHE.get(
            clean_query
        )
    )

    if cached_embedding is not None:
        print(
            "Using cached query embedding."
        )

        return cached_embedding

    embedding = generate_text_embedding(
        clean_query,
        task_type="RETRIEVAL_QUERY",
    )

    QUERY_EMBEDDING_CACHE[
        clean_query
    ] = embedding

    return embedding


def search_museum_knowledge(
    query: str,
    *,
    top_k: int = TOP_K_DEFAULT,
    civilization_id: str | None = None,
) -> list[dict[str, Any]]:
    query_embedding = get_query_embedding(
        query
    )

    filters = [
        "run_id = @run_id",
    ]

    query_parameters = [
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding,
        ),
        bigquery.ScalarQueryParameter(
            "top_k",
            "INT64",
            top_k,
        ),
        bigquery.ScalarQueryParameter(
            "run_id",
            "STRING",
            run_id,
        ),
    ]

    if civilization_id is not None:
        filters.append(
            "civilization_id = @civilization_id"
        )

        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "civilization_id",
                "STRING",
                civilization_id,
            )
        )

    where_sql = (
        "WHERE "
        + " AND ".join(filters)
    )

    sql = f"""
    SELECT
      base.chunk_id AS chunk_id,
      base.run_id AS run_id,
      base.civilization_id AS civilization_id,
      base.document_type AS document_type,
      base.title AS title,
      base.chunk_number AS chunk_number,
      base.global_chunk_number AS global_chunk_number,
      base.chunk_text AS chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
        {where_sql}
      ),
      'embedding',
      (
        SELECT
          @query_embedding AS embedding
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    query_job = bigquery_client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=query_parameters,
        ),
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "run_id": row.run_id,
            "civilization_id": (
                row.civilization_id
            ),
            "document_type": (
                row.document_type
            ),
            "title": row.title,
            "chunk_number": row.chunk_number,
            "global_chunk_number": (
                row.global_chunk_number
            ),
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [46]:
museum_question = """
Which civilization has the strongest combination of archaeological
evidence, historically plausible reconstruction, visual quality,
cross-modal consistency and educational value?

Identify unsupported or speculative elements that should be clearly
marked for museum visitors.
"""

semantic_results = search_museum_knowledge(
    museum_question,
    top_k=10,
)

print("QUERY:")
print(museum_question)

print("\nRESULTS:", len(semantic_results))

for result in semantic_results:
    print("=" * 100)

    print(
        "Distance:",
        round(
            result["distance"],
            4,
        ),
    )

    print(
        "Civilization:",
        result["civilization_id"],
    )

    print(
        "Document type:",
        result["document_type"],
    )

    print(
        "Title:",
        result["title"],
    )

    print(
        "Chunk ID:",
        result["chunk_id"],
    )

    print(
        result["chunk_text"][:900]
    )

QUERY:

Which civilization has the strongest combination of archaeological
evidence, historically plausible reconstruction, visual quality,
cross-modal consistency and educational value?

Identify unsupported or speculative elements that should be clearly
marked for museum visitors.


RESULTS: 10
Distance: 0.2753
Civilization: zephyria_02
Document type: media_asset_card
Title: civilization_image - The Zephyrian Astrolatry
Chunk ID: c57fa99d-d634-481b-a63b-5e125aadbd79
tary cinematography, historically plausible materials, realistic weathering, physically believable engineering, atmospheric natural light and high-detail environmental composition. Strict requirements: - native 4K output - 16:9 landscape composition - no upscaling - use only elements permitted by the visual contract - do not visualize excluded low-confidence claims - no unsupported technology - no fantasy magic - no readable modern text - no logos - no copied historical monuments - no graphic human remains - premium museu

In [47]:
EXHIBITION_RECOMMENDATION_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "recommendation_title",
        "executive_summary",
        "recommended_civilization_order",
        "strongest_civilization_id",
        "strongest_civilization_reason",
        "evidence_quality_observations",
        "media_quality_observations",
        "educational_sequence",
        "visitor_warnings",
        "next_iteration_actions",
        "source_chunk_ids",
        "media_uris_to_use",
    ],
    "properties": {
        "recommendation_title": {
            "type": "STRING",
        },
        "executive_summary": {
            "type": "STRING",
        },
        "recommended_civilization_order": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "strongest_civilization_id": {
            "type": "STRING",
        },
        "strongest_civilization_reason": {
            "type": "STRING",
        },
        "evidence_quality_observations": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "media_quality_observations": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "educational_sequence": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "visitor_warnings": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "next_iteration_actions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "source_chunk_ids": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "media_uris_to_use": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
    },
}

print("Exhibition recommendation schema created.")

Exhibition recommendation schema created.


In [48]:
def build_rag_context(
    results: list[dict[str, Any]],
) -> str:
    context_parts = []

    for source_number, result in enumerate(
        results,
        start=1,
    ):
        context_parts.append(
            "\n".join(
                [
                    f"[SOURCE {source_number}]",
                    f"Chunk ID: {result['chunk_id']}",
                    (
                        f"Civilization ID: "
                        f"{result['civilization_id']}"
                    ),
                    (
                        f"Document type: "
                        f"{result['document_type']}"
                    ),
                    f"Title: {result['title']}",
                    (
                        f"Distance: "
                        f"{result['distance']:.4f}"
                    ),
                    f"Text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(
        context_parts
    )


def generate_exhibition_recommendation(
    question: str,
    *,
    top_k: int = TOP_K_DEFAULT,
) -> tuple[
    ExhibitionRecommendation,
    list[dict[str, Any]],
    str,
]:
    results = search_museum_knowledge(
        question,
        top_k=top_k,
    )

    rag_context = build_rag_context(
        results
    )

    allowed_civilization_ids = [
        civilization.civilization_id
        for civilization
        in civilization_portfolio.civilizations
    ]

    allowed_chunk_ids = [
        result["chunk_id"]
        for result in results
    ]

    available_media = [
        {
            "civilization_id": (
                row["civilization_id"]
            ),
            "media_type": row["media_type"],
            "gcs_uri": row["gcs_uri"],
            "native_4k_verified": (
                row["native_4k_verified"]
            ),
        }
        for row in all_media_rows
    ]

    allowed_media_uris = [
        row["gcs_uri"]
        for row in all_media_rows
    ]

    prompt = f"""
You are the lead curator and archaeological ethics reviewer.

Visitor question:
{question}

Retrieved museum evidence:
{rag_context}

Allowed civilization IDs:
{json.dumps(
    allowed_civilization_ids,
    indent=2,
)}

Allowed chunk IDs:
{json.dumps(
    allowed_chunk_ids,
    indent=2,
)}

Available media:
{json.dumps(
    available_media,
    indent=2,
    ensure_ascii=False,
)}

Rules:
- remain grounded in the retrieved evidence
- use every civilization ID exactly once in the recommended order
- strongest_civilization_id must be allowed
- use only allowed source chunk IDs
- use only available media URIs
- distinguish archaeological evidence from interpretation
- identify speculative material that needs a visitor warning
- account for hallucination risk
- account for native 4K verification
- recommend a practical educational exhibition sequence
- return only valid JSON matching the schema
"""

    raw_recommendation, recommendation_model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=(
                EXHIBITION_RECOMMENDATION_SCHEMA
            ),
            operation_name=(
                "Generate exhibition recommendation"
            ),
            temperature=0.2,
        )
    )

    recommendation = (
        ExhibitionRecommendation.model_validate(
            raw_recommendation
        )
    )

    allowed_civilization_set = set(
        allowed_civilization_ids
    )

    if (
        set(
            recommendation.recommended_civilization_order
        )
        != allowed_civilization_set
    ):
        raise ValueError(
            "The recommended order must contain "
            "all civilizations exactly once."
        )

    if (
        recommendation.strongest_civilization_id
        not in allowed_civilization_set
    ):
        raise ValueError(
            "Invalid strongest_civilization_id."
        )

    invalid_chunk_ids = (
        set(recommendation.source_chunk_ids)
        - set(allowed_chunk_ids)
    )

    invalid_media_uris = (
        set(recommendation.media_uris_to_use)
        - set(allowed_media_uris)
    )

    if invalid_chunk_ids:
        raise ValueError(
            f"Invalid chunk IDs: "
            f"{invalid_chunk_ids}"
        )

    if invalid_media_uris:
        raise ValueError(
            f"Invalid media URIs: "
            f"{invalid_media_uris}"
        )

    return (
        recommendation,
        results,
        recommendation_model_id,
    )


(
    exhibition_recommendation,
    recommendation_sources,
    recommendation_model_id,
) = generate_exhibition_recommendation(
    museum_question,
    top_k=10,
)

print(
    "Recommendation model:",
    recommendation_model_id,
)

print(
    exhibition_recommendation.model_dump_json(
        indent=2
    )
)

Using cached query embedding.
Operation: Generate exhibition recommendation
Trying reasoning model: gemini-3.5-flash
Model failed: gemini-3.5-flash
Unterminated string starting at: line 1 column 2975 (char 2974)
Operation: Generate exhibition recommendation
Trying reasoning model: gemini-2.5-pro
Structured response generated successfully.
Model: gemini-2.5-pro
Recommendation model: gemini-2.5-pro
{
  "recommendation_title": "The Zephyrian Astrolatry: A Model of Evidence-Based Reconstruction",
  "executive_summary": "The Zephyrian Astrolatry (zephyria_02) demonstrates the strongest combination of archaeological evidence, plausible reconstruction, and educational value. Its visual assets are grounded in explicit, high-confidence evidence claims and verified for cross-modal consistency. The other civilizations, while visually compelling, lack the same degree of documented evidentiary support in the provided materials and serve as examples of more speculative interpretation.",
  "recommend

In [49]:
final_manifest = {
    "manifest_id": str(uuid4()),
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
    "notebook": (
        "75_w_google_cloud_lost_civilizations_"
        "4k_museum_studio.ipynb"
    ),
    "museum_brief": museum_brief.model_dump(
        mode="json"
    ),
    "civilization_portfolio": (
        civilization_portfolio.model_dump(
            mode="json"
        )
    ),
    "claim_ledger_gcs_uri": (
        claim_ledger_gcs_uri
    ),
    "visual_contracts": visual_contracts,
    "museum_dialogues": {
        civilization_id: dialogue.model_dump(
            mode="json"
        )
        for civilization_id, dialogue
        in museum_dialogues.items()
    },
    "media_verifications": {
        civilization_id: verification.model_dump(
            mode="json"
        )
        for civilization_id, verification
        in media_verifications.items()
    },
    "exhibition_recommendation": (
        exhibition_recommendation.model_dump(
            mode="json"
        )
    ),
    "media_assets": all_media_rows,
    "model_selection": {
        **model_selection,
        "recommendation_model": (
            recommendation_model_id
        ),
    },
    "asset_counts": {
        "images": len(image_rows),
        "audio_files": len(audio_rows),
        "videos": len(video_rows),
        "total_assets": len(all_media_rows),
    },
    "resolution_policy": {
        "required_image_resolution": (
            IMAGE_RESOLUTION
        ),
        "required_video_resolution": (
            VIDEO_RESOLUTION
        ),
        "minimum_width": MIN_4K_WIDTH,
        "minimum_height": MIN_4K_HEIGHT,
        "upscaling_used": False,
        "downscaling_used": False,
    },
    "local_media_files_saved": False,
}

manifest_json = json.dumps(
    final_manifest,
    indent=2,
    ensure_ascii=False,
    default=str,
)

manifest_blob_name = (
    f"{GCS_MANIFEST_PREFIX}/"
    f"{run_id}/"
    "lost_civilizations_4k_manifest.json"
)

manifest_gcs_uri = upload_text_to_gcs(
    manifest_json,
    blob_name=manifest_blob_name,
    content_type="application/json",
)

recommendation_row = {
    "recommendation_id": str(uuid4()),
    "run_id": run_id,
    "question": museum_question,
    "recommendation_json": (
        exhibition_recommendation.model_dump_json(
            indent=2
        )
    ),
    "used_chunk_ids": (
        exhibition_recommendation.source_chunk_ids
    ),
    "used_media_uris": (
        exhibition_recommendation.media_uris_to_use
    ),
    "manifest_gcs_uri": manifest_gcs_uri,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
}

recommendation_load_result = (
    batch_load_rows_to_bigquery(
        rows=[recommendation_row],
        table_ref=recommendation_table_ref,
        schema=recommendation_schema,
        table_name=RECOMMENDATION_TABLE_ID,
        run_id=run_id,
    )
)

print("Manifest:")
print(manifest_gcs_uri)

print("\nRecommendation load:")
print(recommendation_load_result)

{'table_name': 'exhibition_recommendations', 'rows_loaded': 1, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/exhibition_recommendations.ndjson', 'job_id': 'd8f15054-a4d0-41e5-a015-84b0793b7be3', 'elapsed_seconds': 2.509}
Manifest:
gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/manifests/a232acc3-9195-4c7f-8513-bb0f4be272fe/lost_civilizations_4k_manifest.json

Recommendation load:
{'table_name': 'exhibition_recommendations', 'rows_loaded': 1, 'staging_uri': 'gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/staging/a232acc3-9195-4c7f-8513-bb0f4be272fe/exhibition_recommendations.ndjson', 'job_id': 'd8f15054-a4d0-41e5-a015-84b0793b7be3', 'elapsed_seconds': 2.509}


In [50]:
claim_dashboard_df = pd.DataFrame(
    [
        {
            "civilization_id": row["civilization_id"],
            "claim_id": row["claim_id"],
            "claim_type": row["claim_type"],
            "confidence": row["confidence"],
            "used_for_generation": (
                row["confidence"]
                >= CLAIM_CONFIDENCE_THRESHOLD
            ),
            "statement": row["statement"],
        }
        for row in claim_rows
    ]
).sort_values(
    by=[
        "civilization_id",
        "confidence",
    ],
    ascending=[
        True,
        False,
    ],
)

display(
    claim_dashboard_df
)


quality_rows = []

for civilization in civilization_portfolio.civilizations:
    verification = media_verifications[
        civilization.civilization_id
    ]

    quality_rows.append(
        {
            "civilization_id": (
                civilization.civilization_id
            ),
            "civilization_name": (
                civilization.civilization_name
            ),
            "environment_type": (
                civilization.environment_type
            ),
            "image_claim_fidelity": (
                verification.image_claim_fidelity_score
            ),
            "video_claim_fidelity": (
                verification.video_claim_fidelity_score
            ),
            "audio_evidence_fidelity": (
                verification.audio_evidence_fidelity_score
            ),
            "cross_modal_consistency": (
                verification.cross_modal_consistency_score
            ),
            "historical_plausibility": (
                verification.historical_plausibility_score
            ),
            "hallucination_risk": (
                verification.hallucination_risk
            ),
            "total_score": (
                verification.image_claim_fidelity_score
                + verification.video_claim_fidelity_score
                + verification.audio_evidence_fidelity_score
                + verification.cross_modal_consistency_score
                + verification.historical_plausibility_score
            ),
        }
    )

quality_dashboard_df = pd.DataFrame(
    quality_rows
).sort_values(
    by="total_score",
    ascending=False,
)

display(
    quality_dashboard_df
)

,civilization_id,claim_id,claim_type,confidence,used_for_generation,statement
15,pyroclast_03,pyroclast_03-claim-04,environment,0.98,True,The valley experienced frequent ash-fall event...
14,pyroclast_03,pyroclast_03-claim-03,infrastructure,0.95,True,An extensive network of stone aqueducts was co...
13,pyroclast_03,pyroclast_03-claim-02,architecture,0.90,True,Residential structures utilized thick basalt b...
12,pyroclast_03,pyroclast_03-claim-01,artifact,0.85,True,The Pyralis Hegemony crafted highly specialize...
16,pyroclast_03,pyroclast_03-claim-05,social_practice,0.80,True,The Hegemony engaged in ritual bathing and com...
18,pyroclast_03,pyroclast_03-claim-07,artifact,0.75,True,Heavy bronze protective masks with glass-like ...
17,pyroclast_03,pyroclast_03-claim-06,uncertain_hypothesis,0.35,False,The Hegemony may have utilized sulfur-based co...
1,thalassor_01,thalassor_01-claim-02,architecture,0.90,True,Domed basalt dwellings built directly into the...
4,thalassor_01,thalassor_01-claim-05,social_practice,0.88,True,Ritualistic burial of the dead in deep volcani...
0,thalassor_01,thalassor_01-claim-01,artifact,0.85,True,Weighted bronze diving helmets with integrated...


,civilization_id,civilization_name,environment_type,image_claim_fidelity,video_claim_fidelity,audio_evidence_fidelity,cross_modal_consistency,historical_plausibility,hallucination_risk,total_score
1,zephyria_02,The Zephyrian Astrolatry,desert_astronomer_city,10,10,10,10,9,low,49
2,pyroclast_03,The Pyralis Hegemony,geothermal_mountain_city,10,10,10,10,9,low,49
0,thalassor_01,The Thalassocracy of Nyxalor,submerged_ocean_city,10,10,10,10,8,low,48


In [51]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "global_location": GLOBAL_LOCATION,
    "video_location": VIDEO_LOCATION,
    "bucket_location": BUCKET_LOCATION,
    "bigquery_location": dataset.location,
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
    "exhibition_title": (
        civilization_portfolio.exhibition_title
    ),
    "models": {
        **model_selection,
        "recommendation_model": (
            recommendation_model_id
        ),
    },
    "counts": {
        "civilizations": len(civilization_rows),
        "evidence_claims": len(claim_rows),
        "images": len(image_rows),
        "audio_files": len(audio_rows),
        "videos": len(video_rows),
        "reviews": len(review_rows),
        "knowledge_chunks": len(chunk_rows),
    },
    "resolution_verification": {
        "images": [
            {
                "civilization_id": (
                    row["civilization_id"]
                ),
                "width": row["actual_width"],
                "height": row["actual_height"],
                "native_4k_verified": (
                    row["native_4k_verified"]
                ),
            }
            for row in image_rows
        ],
        "videos": [
            {
                "civilization_id": (
                    row["civilization_id"]
                ),
                "width": row["actual_width"],
                "height": row["actual_height"],
                "native_4k_verified": (
                    row["native_4k_verified"]
                ),
            }
            for row in video_rows
        ],
        "upscaling_used": False,
        "downscaling_used": False,
    },
    "performance": {
        "embedding_seconds": round(
            embedding_seconds,
            2,
        ),
    },
    "cloud_storage": {
        "claim_ledger_gcs_uri": (
            claim_ledger_gcs_uri
        ),
        "manifest_gcs_uri": manifest_gcs_uri,
        "image_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_IMAGE_PREFIX}/{run_id}"
        ),
        "audio_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_AUDIO_PREFIX}/{run_id}"
        ),
        "video_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_VIDEO_PREFIX}/{run_id}"
        ),
        "local_media_files_saved": False,
    },
    "bigquery": {
        "run_table": run_table_ref,
        "civilization_table": (
            civilization_table_ref
        ),
        "claim_table": claim_table_ref,
        "media_table": media_table_ref,
        "review_table": review_table_ref,
        "chunk_table": chunk_table_ref,
        "recommendation_table": (
            recommendation_table_ref
        ),
    },
    "exhibition_recommendation": (
        exhibition_recommendation.model_dump(
            mode="json"
        )
    ),
    "quality_dashboard": (
        quality_dashboard_df.to_dict(
            orient="records"
        )
    ),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{run_id}/"
    "summary.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Summary:")
print(summary_gcs_uri)

Summary:
gs://leafy-guide-497515-m4-vector-assets/lost-civilizations-4k/summaries/a232acc3-9195-4c7f-8513-bb0f4be272fe/summary.json
